# Fine-tuning: Test-subset: BCI Competition III Data II (Subj B)

## Цель эксперимента

Провести финальную честную оценку стратегий обучения на **Test-subject B BCI Competition III**, не использовавшейся для выбора гиперпараметров и стратегий.

Сравниваются:

- scratch
- full fine-tuning (`full_ft`)
- low LR encoder (`low_lr_encoder`)
- partial fine-tuning (`partial_ft`)
- warmup fine-tuning (`warmup`)

---

## Фиксированные условия

Все эксперименты проводятся при одинаковых настройках:

- `lr_encoder = 3e-5`
- `lr_head = 3e-4`
- `weight_decay = 1e-3`
- `warmup_epochs = 3`

Важно:

- гиперпараметры **не подбираются**
- split’ы фиксированы
- нормализация считается только по `Calib_p` конкретного субъекта
- `test_rest` не используется при обучении

---

## Протокол

Для каждого субъекта и каждого уровня калибровки

\[
p \in \{0, 10, 20, 40, 60, 100\}
\]

выполняется:

1. формирование `Calib_p`
2. `train/val` split внутри `Calib_p` (для `p > 0`)
3. обучение модели
4. early stopping по `val_loss`
5. оценка на фиксированном `test_rest`

---

## Метрики

Сохраняются:

- ROC-AUC
- Accuracy
- Binary F1
- Precision
- Recall
- FDR

---

## Архитектура

- Encoder: `UNet1DEncoder`
- Head: Global Average Pooling по времени + Linear (512 → 2)
- Loss: CrossEntropy

---

## Важно про Test

Этот ноутбук используется для **финальной оценки**.  
Результаты из него будут использоваться для итоговых графиков, статистического сравнения и выводов в дипломе.

## 1. Импорты

In [1]:
import os
import json
import math
import random
import gc
from pathlib import Path
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

## 2. Global config

In [2]:
CONFIG = {
    "seed": 42,
    
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pin_memory": torch.cuda.is_available(),

    "data_root": Path("/kaggle/input/datasets/taisiyaglazova"),
    "encoder_checkpoint": Path("/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt"),
    "results_root": Path("/kaggle/working/stage5_block2_results"),

    "batch_size": 64,
    "num_workers": 2,

    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,

    "lr": 1e-4,
    "weight_decay": 1e-4,

    "fallback_p_for_zero": 10,
    "val_ratio": 0.2,

    "p_list": [0, 10, 20, 40, 60, 100],
    "scenarios": ["scratch", "ssl_ft"],

    "save_predictions": True,
    "save_history": True,
}

In [3]:
# Выходные папки
(CONFIG["results_root"] / "history").mkdir(parents=True, exist_ok=True)
(CONFIG["results_root"] / "predictions").mkdir(parents=True, exist_ok=True)
(CONFIG["results_root"] / "checkpoints").mkdir(parents=True, exist_ok=True)
(CONFIG["results_root"] / "logs").mkdir(parents=True, exist_ok=True)

## Config для Dev-tuning

In [4]:
# ----------------------------
# 1) Fixed experimental setup
# ----------------------------
DEV_SUBJECTS = [
    'subj_054', 
    'subj_065', 
    'subj_090', 
    'subj_094'
]

DEV_GROUP = "benchmark"   # или "train" — зафиксируем позже, когда выберем субъектов
DEV_P_LIST = [10, 40, 100]
DEV_SEED = 42

FT_SCENARIOS = [
    "full_ft",
    "low_lr_encoder",
    "partial_ft",
    "warmup",
]

PRIMARY_METRIC = "auc"
CONTROL_METRIC = "f1"

# ----------------------------
# 2) Phase 1: coarse selection
# ----------------------------
PHASE1_CONFIG = {
    "lr_encoder": 1e-5,
    "lr_head": 1e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 3,      # используется только для warmup
}

# ----------------------------
# 3) Phase 2: mini tuning grid
# ----------------------------
PHASE2_GRID = {
    "lr_encoder": [1e-5, 3e-5],
    "lr_head": [1e-4, 3e-4],
    "weight_decay": [1e-4, 1e-3],
    "warmup_epochs": [3],    # пока фиксируем
}

# ----------------------------
# 4) Scenario-specific setup
# ----------------------------
SCENARIO_CONFIGS = {
    "full_ft": {
        "description": "Train full encoder + head",
        "use_discriminative_lr": False,
        "trainable_mode": "full",
        "use_warmup": False,
    },
    "low_lr_encoder": {
        "description": "Train full encoder + head with lower LR for encoder",
        "use_discriminative_lr": True,
        "trainable_mode": "full",
        "use_warmup": False,
    },
    "partial_ft": {
        "description": "Train only down4 + head",
        "use_discriminative_lr": True,
        "trainable_mode": "down4_only",
        "use_warmup": False,
    },
    "warmup": {
        "description": "Stage 1: head only, Stage 2: full FT with lower encoder LR",
        "use_discriminative_lr": True,
        "trainable_mode": "full",
        "use_warmup": True,
    },
}

# ----------------------------
# 5) Output folders for Block 4
# ----------------------------
BLOCK4_TAG = "stage5_block4_dev_tuning"

BLOCK4_DIR = Path(CONFIG["results_root"]) / BLOCK4_TAG
BLOCK4_PHASE1_DIR = BLOCK4_DIR / "phase1"
BLOCK4_PHASE2_DIR = BLOCK4_DIR / "phase2"

BLOCK4_HISTORY_DIR = BLOCK4_DIR / "history"
BLOCK4_PREDS_DIR = BLOCK4_DIR / "predictions"
BLOCK4_TABLES_DIR = BLOCK4_DIR / "tables"
BLOCK4_FIGURES_DIR = BLOCK4_DIR / "figures"

for d in [
    BLOCK4_DIR,
    BLOCK4_PHASE1_DIR,
    BLOCK4_PHASE2_DIR,
    BLOCK4_HISTORY_DIR,
    BLOCK4_PREDS_DIR,
    BLOCK4_TABLES_DIR,
    BLOCK4_FIGURES_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 6) Helpers to build grids
# ----------------------------
def make_phase2_param_grid(grid_dict):
    keys = list(grid_dict.keys())
    values = [grid_dict[k] for k in keys]
    return [dict(zip(keys, combo)) for combo in product(*values)]

PHASE2_PARAM_GRID = make_phase2_param_grid(PHASE2_GRID)

# ----------------------------
# 7) Sanity print
# ----------------------------
print("Block 4 configuration loaded.")
print(f"DEV subjects: {DEV_SUBJECTS}")
print(f"DEV group   : {DEV_GROUP}")
print(f"p list      : {DEV_P_LIST}")
print(f"scenarios   : {FT_SCENARIOS}")
print(f"phase2 grid size: {len(PHASE2_PARAM_GRID)}")
print(f"results dir : {BLOCK4_DIR}")

Block 4 configuration loaded.
DEV subjects: ['subj_054', 'subj_065', 'subj_090', 'subj_094']
DEV group   : benchmark
p list      : [10, 40, 100]
scenarios   : ['full_ft', 'low_lr_encoder', 'partial_ft', 'warmup']
phase2 grid size: 8
results dir : /kaggle/working/stage5_block2_results/stage5_block4_dev_tuning


## 3. Воспроизводимость

In [5]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [6]:
set_seed(CONFIG["seed"])
print("Device:", CONFIG["device"])

Device: cuda


## 4. Пути

In [7]:
# Пути к датасетам
DATASETS = {
    "bigp3_train": CONFIG["data_root"] / "bigp3bci-downstream-train",
    "bigp3_benchmark": CONFIG["data_root"] / "bigp3bci-downstream-benchmark",
    "bcicomp3": CONFIG["data_root"] / "bcicompiii-ds2",  
}

In [8]:
GROUPS = ["train", "benchmark", "bcicomp3"]

In [9]:
# Универсальные функции путей
def get_epochs_path(subject_id: str, group: str) -> Path:
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        path = DATASETS["bigp3_train"] / "train" / f"{subject_id}.npz"
    elif group == "benchmark":
        path = DATASETS["bigp3_benchmark"]/ "benchmark" / f"{subject_id}.npz"
    elif group == "bcicomp3":
        path = DATASETS["bcicomp3"] / "epochs" / f"{subject_id}_train_epochs_v1.npz"
    
    return path


def get_split_path(subject_id: str, group: str) -> Path:
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        path = DATASETS["bigp3_train"] / "splits" / "train" / f"{subject_id}.json"
    elif group == "benchmark":
        path = DATASETS["bigp3_benchmark"] / "splits" / "benchmark" / f"{subject_id}.json"
    elif group == "bcicomp3":
        path = DATASETS["bcicomp3"] / "splits" / f"{subject_id}_time30_seed42_v1.json"
        
    return path


def get_stats_path(subject_id: str, p: int, group: str) -> Path:
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        path = DATASETS["bigp3_train"] / "stats" / "train" / f"{subject_id}_p{p}.npz"
    elif group == "benchmark":
        path = DATASETS["bigp3_benchmark"] / "stats" / "benchmark" / f"{subject_id}_p{p}.npz"
    elif group == "bcicomp3":
        path = DATASETS["bcicomp3"] / "stats" / f"{subject_id}_time30_seed42_p{p}_v1.npz"
        
    return path

In [10]:
# Автореестр субъектов
def list_subject_ids(group: str):
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        data_dir = DATASETS["bigp3_train"] / "train"
    elif group == "benchmark":
        data_dir = DATASETS["bigp3_benchmark"]/ "benchmark"

    subject_ids = sorted([p.stem for p in data_dir.glob("*.npz")])
    return subject_ids

In [11]:
# Сборка реестра
SUBJECT_REGISTRY = {
    "train": list_subject_ids("train"),
    "benchmark": list_subject_ids("benchmark"),
}

print("Train subjects:", len(SUBJECT_REGISTRY["train"]))
print("Benchmark subjects:", len(SUBJECT_REGISTRY["benchmark"]))
print("Example train:", SUBJECT_REGISTRY["train"][:5])
print("Example benchmark:", SUBJECT_REGISTRY["benchmark"][:5])


Train subjects: 93
Benchmark subjects: 65
Example train: ['subj_000', 'subj_001', 'subj_002', 'subj_003', 'subj_004']
Example benchmark: ['subj_051', 'subj_052', 'subj_053', 'subj_054', 'subj_055']


In [12]:
# Валидация существования файлов
def check_subject_files(subject_id: str, group: str, p_list=(10, 20, 40, 60, 100)):
    report = {
        "epochs": get_epochs_path(subject_id, group).exists(),
        "split": get_split_path(subject_id, group).exists(),
        "stats": {p: get_stats_path(subject_id, p, group).exists() for p in p_list},
    }
    return report

example_subj = SUBJECT_REGISTRY["train"][80]
check_subject_files(example_subj, "train")

{'epochs': True,
 'split': True,
 'stats': {10: True, 20: True, 40: True, 60: True, 100: True}}

## Конфиг для full Dev run

In [13]:
FINAL_DEV_CONFIG = {
    "subjects": SUBJECT_REGISTRY["train"],   # или твой Dev-список, если он уже зафиксирован отдельно
    "group": "train",
    "p_list": [10, 20, 40, 60, 100],
    "ft_strategy_list": ["full_ft", "low_lr_encoder", "partial_ft", "warmup"],
    "seed_list": [42],

    "lr_encoder": 3e-5,
    "lr_head": 3e-4,
    "weight_decay": 1e-3,
    "warmup_epochs": 3,
}

## Конфиг для FT-сценариев на Test

In [14]:
DEV_SUBJECTS = [
    'subj_054', 
    'subj_065', 
    'subj_090', 
    'subj_094'
]

In [15]:
ALL_BENCHMARK_SUBJECTS = SUBJECT_REGISTRY["benchmark"]

FILTERED_BENCHMARK_SUBJECTS = [
    s for s in ALL_BENCHMARK_SUBJECTS
    if s not in DEV_SUBJECTS
]

print("Original benchmark subjects:", len(ALL_BENCHMARK_SUBJECTS))
print("After exclusion:", len(FILTERED_BENCHMARK_SUBJECTS))
print("Excluded:", DEV_SUBJECTS)

Original benchmark subjects: 65
After exclusion: 61
Excluded: ['subj_054', 'subj_065', 'subj_090', 'subj_094']


In [16]:
FINAL_TEST_CONFIG = {
    "subjects": FILTERED_BENCHMARK_SUBJECTS,
    "group": "benchmark",

    "p_list": [0, 10, 20, 40, 60, 100],

    "ft_strategy_list": ["full_ft", "low_lr_encoder", "partial_ft", "warmup"],
    "seed_list": [42],

    "lr_encoder": 3e-5,
    "lr_head": 3e-4,
    "weight_decay": 1e-3,
    "warmup_epochs": 3,
}

## Конфиг для Scratch на Test

In [17]:
FINAL_TEST_SCRATCH_CONFIG = {
    "subjects": FILTERED_BENCHMARK_SUBJECTS,
    "group": "benchmark",

    "p_list": [0, 10, 20, 40, 60, 100],
    "seed_list": [42],

    "lr_encoder": 3e-5,
    "lr_head": 3e-4,
    "weight_decay": 1e-3,
}

## Пути для FT и Scratch на Test

In [18]:
FINAL_TEST_TAG = "stage5_final_eval_test_SubjB_all_strategies"
FINAL_TEST_DIR = Path(CONFIG["results_root"]) / FINAL_TEST_TAG
FINAL_TEST_DIR.mkdir(parents=True, exist_ok=True)

In [19]:
FINAL_TEST_FT_DIR = FINAL_TEST_DIR / "ft_strategies"
FINAL_TEST_SCRATCH_DIR = FINAL_TEST_DIR / "scratch"

FINAL_TEST_FT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_TEST_SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

In [20]:
# Проверка
print("FINAL_TEST_DIR:", FINAL_TEST_DIR)
print("FINAL_TEST_FT_DIR:", FINAL_TEST_FT_DIR)
print("FINAL_TEST_SCRATCH_DIR:", FINAL_TEST_SCRATCH_DIR)

print("Group:", FINAL_TEST_CONFIG["group"])
print("N benchmark subjects:", len(FINAL_TEST_CONFIG["subjects"]))
print("p_list:", FINAL_TEST_CONFIG["p_list"])
print("ft_strategy_list:", FINAL_TEST_CONFIG["ft_strategy_list"])
print("seed_list:", FINAL_TEST_CONFIG["seed_list"])

print("lr_encoder:", FINAL_TEST_CONFIG["lr_encoder"])
print("lr_head:", FINAL_TEST_CONFIG["lr_head"])
print("weight_decay:", FINAL_TEST_CONFIG["weight_decay"])
print("warmup_epochs:", FINAL_TEST_CONFIG["warmup_epochs"])

FINAL_TEST_DIR: /kaggle/working/stage5_block2_results/stage5_final_eval_test_SubjB_all_strategies
FINAL_TEST_FT_DIR: /kaggle/working/stage5_block2_results/stage5_final_eval_test_SubjB_all_strategies/ft_strategies
FINAL_TEST_SCRATCH_DIR: /kaggle/working/stage5_block2_results/stage5_final_eval_test_SubjB_all_strategies/scratch
Group: benchmark
N benchmark subjects: 61
p_list: [0, 10, 20, 40, 60, 100]
ft_strategy_list: ['full_ft', 'low_lr_encoder', 'partial_ft', 'warmup']
seed_list: [42]
lr_encoder: 3e-05
lr_head: 0.0003
weight_decay: 0.001
warmup_epochs: 3


## 5. Core data I/O

In [21]:
# Загрузка эпох
def load_subject_epochs(subject_id: str, group: str):
    path = get_epochs_path(subject_id, group)
    if not path.exists():
        raise FileNotFoundError(f"Epochs file not found: {path}")

    data = np.load(path, allow_pickle=True)

    if "X" not in data or "y" not in data:
        raise KeyError(f"{path} must contain keys 'X' and 'y'. Found: {list(data.keys())}")

    X = data["X"]
    y = data["y"]

    return X, y

In [22]:
# Загрузка split
def load_subject_split(subject_id: str, group: str):
    path = get_split_path(subject_id, group)
    if not path.exists():
        raise FileNotFoundError(f"Split file not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        split = json.load(f)

    # Приводим формат bcicomp3 к стандартному формату pipeline
    if group == "bcicomp3":
        if "indices" not in split:
            raise KeyError("bcicomp3 split must contain top-level key 'indices'")

        idx = split["indices"]

        normalized_split = {
            "calib_pool_idx": idx["calib_pool_idx"],
            "test_rest_idx": idx["test_rest_idx"],
            "calib_idx": idx["calib_idx"],
        }
        return normalized_split

    return split

In [23]:
# Загрузка stats
def load_subject_stats(subject_id: str, p: int, group: str):
    path = get_stats_path(subject_id, p, group)
    if not path.exists():
        raise FileNotFoundError(f"Stats file not found: {path}")

    data = np.load(path, allow_pickle=True)

    if "mean" not in data or "std" not in data:
        raise KeyError(f"{path} must contain keys 'mean' and 'std'. Found: {list(data.keys())}")

    mean = data["mean"]
    std = data["std"]

    return mean, std

In [24]:
# Объединённая загрузка bundle
def load_subject_bundle(subject_id: str, p: int, group: str):
    X, y = load_subject_epochs(subject_id, group)
    split = load_subject_split(subject_id, group)

    mean, std = (None, None)
    if p > 0:
        mean, std = load_subject_stats(subject_id, p, group)

    bundle = {
        "subject_id": subject_id,
        "group": group,
        "p": p,
        "X": X,
        "y": y,
        "split": split,
        "mean": mean,
        "std": std,
    }
    return bundle

In [25]:
# # ============================================================
# # CHECK: path/data-loading functions for Subject B (BCICompIII)
# # tolerant to p=0 case where mean/std may be None in bundle
# # ============================================================

# from pathlib import Path
# import json
# import numpy as np

# SUBJECT_ID = "subjB"   # поменяй на "subj_B", если нужно
# GROUP = "bcicomp3"
# P_LIST = [0, 10, 20, 40, 60, 100]


# def _assert_exists(path, name):
#     assert path is not None, f"{name}: returned None"
#     assert Path(path).exists(), f"{name}: path does not exist -> {path}"


# def _check_npz_epochs(path):
#     data = np.load(path, allow_pickle=True)
#     keys = list(data.keys())
#     print(f"[epochs] keys: {keys}")

#     assert "X" in data, "epochs npz must contain key 'X'"
#     assert "y" in data, "epochs npz must contain key 'y'"

#     X = data["X"]
#     y = data["y"]

#     print(f"[epochs] X.shape={X.shape}, X.dtype={X.dtype}")
#     print(f"[epochs] y.shape={y.shape}, y.dtype={y.dtype}")
#     print(f"[epochs] class counts: {dict(zip(*np.unique(y, return_counts=True)))}")

#     assert X.ndim == 3, f"Expected X.ndim == 3, got {X.ndim}"
#     assert X.shape[1] == 14, f"Expected 14 channels, got {X.shape[1]}"
#     assert X.shape[2] == 208, f"Expected 208 timepoints, got {X.shape[2]}"
#     assert y.ndim == 1, f"Expected y.ndim == 1, got {y.ndim}"
#     assert len(X) == len(y), f"Length mismatch: len(X)={len(X)} vs len(y)={len(y)}"

#     return X, y


# def _check_raw_split_file(path):
#     with open(path, "r", encoding="utf-8") as f:
#         split_raw = json.load(f)

#     print(f"[raw split] top-level keys: {list(split_raw.keys())}")
#     assert "indices" in split_raw, "Raw bcicomp3 split must contain top-level key 'indices'"

#     idx = split_raw["indices"]
#     print(f"[raw split] indices keys: {list(idx.keys())}")

#     assert "calib_pool_idx" in idx, "Missing 'calib_pool_idx' in split['indices']"
#     assert "test_rest_idx" in idx, "Missing 'test_rest_idx' in split['indices']"
#     assert "calib_idx" in idx, "Missing 'calib_idx' in split['indices']"

#     return split_raw


# def _check_normalized_split(split_norm, n_samples):
#     print(f"[normalized split] keys: {list(split_norm.keys())}")

#     assert "calib_pool_idx" in split_norm, "Normalized split missing 'calib_pool_idx'"
#     assert "test_rest_idx" in split_norm, "Normalized split missing 'test_rest_idx'"
#     assert "calib_idx" in split_norm, "Normalized split missing 'calib_idx'"

#     calib_pool_idx = np.asarray(split_norm["calib_pool_idx"], dtype=int)
#     test_rest_idx = np.asarray(split_norm["test_rest_idx"], dtype=int)
#     calib_idx_dict = split_norm["calib_idx"]

#     assert np.all((0 <= calib_pool_idx) & (calib_pool_idx < n_samples)), "calib_pool_idx out of range"
#     assert np.all((0 <= test_rest_idx) & (test_rest_idx < n_samples)), "test_rest_idx out of range"

#     inter = np.intersect1d(calib_pool_idx, test_rest_idx)
#     assert len(inter) == 0, f"calib_pool_idx and test_rest_idx overlap: n_overlap={len(inter)}"

#     union = np.union1d(calib_pool_idx, test_rest_idx)
#     assert len(union) == n_samples, (
#         f"calib_pool_idx + test_rest_idx do not cover all samples: "
#         f"covered={len(union)}, total={n_samples}"
#     )

#     print(f"[normalized split] calib_pool={len(calib_pool_idx)}, test_rest={len(test_rest_idx)}")

#     prev = None
#     prev_p = None
#     for p_key in sorted(calib_idx_dict.keys(), key=lambda x: int(x)):
#         idx_p = np.asarray(calib_idx_dict[p_key], dtype=int)

#         assert np.all((0 <= idx_p) & (idx_p < n_samples)), f"calib_idx[{p_key}] out of range"
#         assert np.all(np.isin(idx_p, calib_pool_idx)), f"calib_idx[{p_key}] is not subset of calib_pool_idx"

#         if prev is not None:
#             assert np.all(np.isin(prev, idx_p)), (
#                 f"Nested property broken: calib_{prev_p} not subset of calib_{p_key}"
#             )

#         print(f"[normalized split] p={p_key}: n={len(idx_p)}")
#         prev = idx_p
#         prev_p = p_key


# def _check_stats_file(path):
#     stats = np.load(path, allow_pickle=True)
#     keys = list(stats.keys())
#     print(f"[stats] keys: {keys}")

#     assert "mean" in stats, "stats npz must contain key 'mean'"
#     assert "std" in stats, "stats npz must contain key 'std'"

#     mean = stats["mean"]
#     std = stats["std"]

#     print(f"[stats] mean.shape={mean.shape}, std.shape={std.shape}")
#     assert mean.shape == (14,), f"Expected mean.shape == (14,), got {mean.shape}"
#     assert std.shape == (14,), f"Expected std.shape == (14,), got {std.shape}"
#     assert np.all(np.isfinite(mean)), "mean contains non-finite values"
#     assert np.all(np.isfinite(std)), "std contains non-finite values"
#     assert np.all(std > 0), "std must be strictly positive"

#     return mean, std


# def _check_bundle(bundle, p):
#     assert isinstance(bundle, dict), "load_subject_bundle must return dict"
#     for k in ["X", "y", "split", "mean", "std"]:
#         assert k in bundle, f"bundle missing key: {k}"

#     X = bundle["X"]
#     y = bundle["y"]
#     split = bundle["split"]
#     mean = bundle["mean"]
#     std = bundle["std"]

#     print(f"[bundle p={p}] X.shape={X.shape}, y.shape={y.shape}")
#     print(f"[bundle p={p}] split keys={list(split.keys())}")

#     assert X.ndim == 3 and X.shape[1:] == (14, 208), f"Unexpected X shape in bundle: {X.shape}"
#     assert y.ndim == 1 and len(y) == len(X), "Unexpected y shape in bundle"

#     _check_normalized_split(split, n_samples=len(X))

#     if p == 0:
#         print(f"[bundle p=0] mean/std may be None by design")
#         if mean is not None:
#             assert mean.shape == (14,), f"Unexpected mean shape in bundle for p=0: {mean.shape}"
#         if std is not None:
#             assert std.shape == (14,), f"Unexpected std shape in bundle for p=0: {std.shape}"
#     else:
#         assert mean is not None, f"bundle mean is None for p={p}"
#         assert std is not None, f"bundle std is None for p={p}"
#         print(f"[bundle p={p}] mean.shape={mean.shape}, std.shape={std.shape}")
#         assert mean.shape == (14,), f"Unexpected mean shape in bundle: {mean.shape}"
#         assert std.shape == (14,), f"Unexpected std shape in bundle: {std.shape}"
#         assert np.all(np.isfinite(mean)), "bundle mean contains non-finite values"
#         assert np.all(np.isfinite(std)), "bundle std contains non-finite values"
#         assert np.all(std > 0), "bundle std must be strictly positive"


# def check_bcicomp3_subjectB_loading(
#     subject_id=SUBJECT_ID,
#     group=GROUP,
#     p_list=P_LIST,
# ):
#     print("=" * 80)
#     print(f"CHECKING SUBJECT: {subject_id} | GROUP: {group}")
#     print("=" * 80)

#     # 1) epochs
#     epochs_path = get_epochs_path(subject_id=subject_id, group=group)
#     print(f"\n[get_epochs_path] -> {epochs_path}")
#     _assert_exists(epochs_path, "get_epochs_path")
#     X, y = _check_npz_epochs(epochs_path)
#     n_samples = len(X)

#     # 2) raw split file
#     split_path = get_split_path(subject_id=subject_id, group=group)
#     print(f"\n[get_split_path] -> {split_path}")
#     _assert_exists(split_path, "get_split_path")
#     _check_raw_split_file(split_path)

#     # 3) normalized split
#     split_norm = load_subject_split(subject_id=subject_id, group=group)
#     print(f"\n[load_subject_split]")
#     _check_normalized_split(split_norm, n_samples=n_samples)

#     # 4) stats files for p > 0
#     for p in p_list:
#         if p == 0:
#             print(f"\n[get_stats_path p=0] skipped (expected fallback behavior)")
#             continue

#         stats_path = get_stats_path(subject_id=subject_id, group=group, p=p)
#         print(f"\n[get_stats_path p={p}] -> {stats_path}")
#         _assert_exists(stats_path, f"get_stats_path(p={p})")
#         _check_stats_file(stats_path)

#     # 5) full bundle
#     for p in p_list:
#         print(f"\n[load_subject_bundle] checking p={p}")
#         bundle = load_subject_bundle(subject_id=subject_id, group=group, p=p)
#         _check_bundle(bundle, p=p)

#     print("\n" + "=" * 80)
#     print("ALL CHECKS PASSED")
#     print("=" * 80)

In [26]:
# check_bcicomp3_subjectB_loading()

## Split внутри Calib_p

In [27]:
# Функции для извлечения индексов
def get_test_indices(split: dict) -> np.ndarray:
    """
    Возвращает индексы test_rest в глобальной индексации subject-level X/y.
    """
    if "test_rest_idx" not in split:
        raise KeyError("split must contain 'test_rest_idx'")
    return np.asarray(split["test_rest_idx"], dtype=np.int64)


def get_calib_pool_indices(split: dict) -> np.ndarray:
    """
    Возвращает индексы calib_pool в глобальной индексации subject-level X/y.
    """
    if "calib_pool_idx" not in split:
        raise KeyError("split must contain 'calib_pool_idx'")
    return np.asarray(split["calib_pool_idx"], dtype=np.int64)


def get_calib_indices(split: dict, p: int) -> np.ndarray:
    """
    Возвращает индексы calib_p в глобальной индексации subject-level X/y.

    Для p=0 возвращает пустой массив.
    """
    if p == 0:
        return np.asarray([], dtype=np.int64)

    if "calib_idx" not in split:
        raise KeyError("split must contain 'calib_idx'")

    calib_idx_dict = split["calib_idx"]

    if not isinstance(calib_idx_dict, dict):
        raise TypeError(f"split['calib_idx'] must be dict, got {type(calib_idx_dict)}")

    if str(p) not in calib_idx_dict:
        raise KeyError(
            f"p={p} not found in split['calib_idx']; available keys: {list(calib_idx_dict.keys())}"
        )

    return np.asarray(calib_idx_dict[str(p)], dtype=np.int64)


def make_train_val_split(
    calib_idx: np.ndarray,
    y: np.ndarray,
    val_ratio: float = 0.2,
    seed: int = 42,
    stratify: bool = True,
):
    """
    Делит calib_idx на train_idx и val_idx.

    Все индексы остаются глобальными относительно X/y конкретного subject.
    """
    calib_idx = np.asarray(calib_idx, dtype=np.int64)

    if len(calib_idx) == 0:
        return (
            np.asarray([], dtype=np.int64),
            np.asarray([], dtype=np.int64),
        )

    if len(calib_idx) < 2:
        raise ValueError("calib_idx must contain at least 2 samples")

    y_calib = y[calib_idx]
    stratify_labels = y_calib if stratify else None

    try:
        train_idx, val_idx = train_test_split(
            calib_idx,
            test_size=val_ratio,
            random_state=seed,
            shuffle=True,
            stratify=stratify_labels,
        )
    except ValueError as e:
        print(f"[WARN] Stratified split failed: {e}")
        print("[WARN] Falling back to non-stratified split.")
        train_idx, val_idx = train_test_split(
            calib_idx,
            test_size=val_ratio,
            random_state=seed,
            shuffle=True,
            stratify=None,
        )

    return (
        np.asarray(train_idx, dtype=np.int64),
        np.asarray(val_idx, dtype=np.int64),
    )


def prepare_run_indices(
    split: dict,
    y: np.ndarray,
    p: int,
    val_ratio: float = 0.2,
    seed: int = 42,
):
    """
    Готовит все индексы для одного запуска.

    Возвращает словарь:
    - calib_pool_idx
    - calib_idx
    - train_idx
    - val_idx
    - test_idx
    """
    calib_pool_idx = get_calib_pool_indices(split)
    test_idx = get_test_indices(split)
    calib_idx = get_calib_indices(split, p)

    calib_pool_set = set(calib_pool_idx.tolist())
    test_set = set(test_idx.tolist())
    calib_set = set(calib_idx.tolist())

    # test и calib_pool не должны пересекаться
    if len(calib_pool_set & test_set) > 0:
        raise ValueError("calib_pool_idx intersects with test_rest_idx")

    # calib_p должен быть подмножеством calib_pool
    if len(calib_set) > 0 and not calib_set.issubset(calib_pool_set):
        raise ValueError("calib_idx is not a subset of calib_pool_idx")

    if p == 0:
        train_idx = np.asarray([], dtype=np.int64)
        val_idx = np.asarray([], dtype=np.int64)
    else:
        train_idx, val_idx = make_train_val_split(
            calib_idx=calib_idx,
            y=y,
            val_ratio=val_ratio,
            seed=seed,
            stratify=True,
        )

    return {
        "calib_pool_idx": calib_pool_idx,
        "calib_idx": calib_idx,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
    }

## Нормализация и нарезка данных по индексам

In [28]:
# Фукции нормализации
def safe_standardize(X: np.ndarray, mean: np.ndarray, std: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    Поканальная z-нормализация массива X формы (N, C, L)
    по mean/std формы (C,).
    """
    X = np.asarray(X, dtype=np.float32)
    mean = np.asarray(mean, dtype=np.float32)
    std = np.asarray(std, dtype=np.float32)

    if X.ndim != 3:
        raise ValueError(f"X must have shape (N, C, L), got {X.shape}")
    if mean.ndim != 1 or std.ndim != 1:
        raise ValueError(f"mean/std must have shape (C,), got mean={mean.shape}, std={std.shape}")
    if X.shape[1] != len(mean) or X.shape[1] != len(std):
        raise ValueError(
            f"Channel mismatch: X has C={X.shape[1]}, mean={len(mean)}, std={len(std)}"
        )

    std_safe = np.maximum(std, eps)
    X_norm = (X - mean[None, :, None]) / std_safe[None, :, None]
    return X_norm.astype(np.float32)


def get_effective_stats(bundle: dict, subject_id: str, group: str, p: int, fallback_p_for_zero: int = 10):
    """
    Возвращает mean/std для данного запуска.
    
    Для p>0 используются stats именно этого p.
    Для p=0 используются fallback-статистики, например p=10.
    """
    if p > 0:
        mean = bundle["mean"]
        std = bundle["std"]
    else:
        mean, std = load_subject_stats(subject_id, fallback_p_for_zero, group)

    if mean is None or std is None:
        raise ValueError(f"Stats are not available for subject={subject_id}, group={group}, p={p}")

    return mean, std

In [29]:
# Функции нарезки по индексам
def slice_by_indices(X: np.ndarray, y: np.ndarray, idx: np.ndarray):
    """
    Возвращает подмножество X/y по глобальным индексам subject-level массива.
    """
    idx = np.asarray(idx, dtype=np.int64)

    if len(idx) == 0:
        X_empty = np.empty((0, X.shape[1], X.shape[2]), dtype=np.float32)
        y_empty = np.empty((0,), dtype=y.dtype)
        return X_empty, y_empty

    return X[idx], y[idx]


def prepare_indexed_arrays(
    bundle: dict,
    indices_dict: dict,
    fallback_p_for_zero: int = 10,
):
    """
    Подготавливает нормализованные массивы:
    - X_train, y_train
    - X_val, y_val
    - X_test, y_test

    bundle должен содержать:
    - subject_id
    - group
    - p
    - X
    - y
    - mean/std (если p>0)
    """
    subject_id = bundle["subject_id"]
    group = bundle["group"]
    p = bundle["p"]

    X = bundle["X"]
    y = bundle["y"]

    mean, std = get_effective_stats(
        bundle=bundle,
        subject_id=subject_id,
        group=group,
        p=p,
        fallback_p_for_zero=fallback_p_for_zero,
    )

    X_norm = safe_standardize(X, mean, std)

    X_train, y_train = slice_by_indices(X_norm, y, indices_dict["train_idx"])
    X_val, y_val = slice_by_indices(X_norm, y, indices_dict["val_idx"])
    X_test, y_test = slice_by_indices(X_norm, y, indices_dict["test_idx"])

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "mean": mean,
        "std": std,
    }

## 6. Dataset / DataLoader

In [30]:
class EEGDataset(Dataset):
    """
    Простой Dataset для EEG-эпох.
    X: np.ndarray формы (N, C, L)
    y: np.ndarray формы (N,)
    """
    def __init__(self, X: np.ndarray, y: np.ndarray):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y)

        if X.ndim != 3:
            raise ValueError(f"X must have shape (N, C, L), got {X.shape}")
        if y.ndim != 1:
            raise ValueError(f"y must have shape (N,), got {y.shape}")
        if len(X) != len(y):
            raise ValueError(f"Length mismatch: len(X)={len(X)}, len(y)={len(y)}")

        self.X = X
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])      # float32, shape (C, L)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y

In [31]:
def make_loader(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
    num_workers: int = 0,
    pin_memory: bool = True,
):
    """
    Создаёт DataLoader для заданных X/y.
    Если массив пустой, возвращает None.
    """
    if len(y) == 0:
        return None

    dataset = EEGDataset(X, y)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )
    return loader

In [32]:
def build_loaders(
    arrays_dict: dict,
    batch_size: int = 64,
    num_workers: int = 0,
    pin_memory: bool = True,
):
    """
    По словарю prepared arrays создаёт:
    - train_loader
    - val_loader
    - test_loader

    Для p=0 train/val будут None.
    """
    train_loader = make_loader(
        X=arrays_dict["X_train"],
        y=arrays_dict["y_train"],
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    val_loader = make_loader(
        X=arrays_dict["X_val"],
        y=arrays_dict["y_val"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    test_loader = make_loader(
        X=arrays_dict["X_test"],
        y=arrays_dict["y_test"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
    }

## 7. Определение модели

In [33]:
class DoubleConv1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Down1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool1d(kernel_size=2, stride=2),
            DoubleConv1D(in_channels, out_channels)
        )

    def forward(self, x):
        return self.block(x)


class Up1D(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.bilinear = bilinear

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="linear", align_corners=True)
            mid_channels = in_channels // 2
            self.conv = DoubleConv1D(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose1d(in_channels, out_channels, kernel_size=2, stride=2)
            self.conv = DoubleConv1D(in_channels, out_channels)

    def forward(self, x1, x2):
        # x1: низ, x2: skip
        x1 = self.up(x1)

        # подгоняем длину, если не совпадает
        diff = x2.size(-1) - x1.size(-1)
        if diff > 0:
            x1 = nn.functional.pad(x1, (diff // 2, diff - diff // 2))
        elif diff < 0:
            x2 = nn.functional.pad(x2, (-diff // 2, -diff - (-diff // 2)))

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet1D_Light(nn.Module):
    """
    U-Net 1D с 4 уровнями даунсемплинга, вдохновлён Hong et al., 2025.
    Каналы: 32 → 64 → 128 → 256, bottleneck 512.
    """
    def __init__(self, n_channels, n_classes, base_ch=32, bilinear=True):
        super().__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        ch1 = base_ch
        ch2 = base_ch * 2
        ch3 = base_ch * 4
        ch4 = base_ch * 8
        bottleneck_ch = base_ch * 16  # 512 при base_ch=32

        # encoder
        self.inc = DoubleConv1D(n_channels, ch1)
        self.down1 = Down1D(ch1, ch2)
        self.down2 = Down1D(ch2, ch3)
        self.down3 = Down1D(ch3, ch4)
        self.down4 = Down1D(ch4, bottleneck_ch)

        # decoder
        self.up1 = Up1D(bottleneck_ch + ch4, ch4, bilinear)
        self.up2 = Up1D(ch4 + ch3, ch3, bilinear)
        self.up3 = Up1D(ch3 + ch2, ch2, bilinear)
        self.up4 = Up1D(ch2 + ch1, ch1, bilinear)
        self.outc = OutConv1D(ch1, n_classes)

    def encode(self, x):
        x1 = self.inc(x)     # (N, ch1, L)
        x2 = self.down1(x1)  # (N, ch2, L/2)
        x3 = self.down2(x2)  # (N, ch3, L/4)
        x4 = self.down3(x3)  # (N, ch4, L/8)
        x5 = self.down4(x4)  # (N, bottleneck_ch, L/16)
        return x1, x2, x3, x4, x5

    def decode(self, x1, x2, x3, x4, x5):
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

    def forward(self, x):
        x1, x2, x3, x4, x5 = self.encode(x)
        logits = self.decode(x1, x2, x3, x4, x5)
        return logits, x5  # x5 — bottleneck

class UNet1DEncoder(nn.Module):
    """
    Обёртка над encoder-частью обученного U-Net.
    На вход:  x ∈ R^{B×C×L}
    На выход: bottleneck-фичи x5 ∈ R^{B×C_bottleneck×L_reduced}
    """
    def __init__(self, unet_model: nn.Module):
        super().__init__()
        # просто переиспользуем уже обученные блоки
        self.inc = unet_model.inc
        self.down1 = unet_model.down1
        self.down2 = unet_model.down2
        self.down3 = unet_model.down3
        self.down4 = unet_model.down4

    def forward(self, x):
        # это ровно то, что делал encode() в полном U-Net
        x1 = self.inc(x)    # (B, ch1, L)
        x2 = self.down1(x1) # (B, ch2, L/2)
        x3 = self.down2(x2) # (B, ch3, L/4)
        x4 = self.down3(x3) # (B, ch4, L/8)
        x5 = self.down4(x4) # (B, bottleneck_ch, L/16)
        return x5

class ERPHead(nn.Module):
    """
    Минимальная классификационная голова для P300.
    Вход:  (B, F, T)
    Выход: (B, 2) logits
    """
    def __init__(self, in_features=512, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, z):
        # z: (B, F, T)
        z = z.mean(dim=-1)   # global average pooling → (B, F)
        logits = self.fc(z)  # (B, 2)
        return logits
    
class P300Model(nn.Module):
    """
    Полная downstream модель:
    x → encoder → head → logits
    """
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, x):
        z = self.encoder(x)
        logits = self.head(z)
        return logits

In [34]:
# Загрузка весов
def load_encoder_checkpoint_into_model_encoder(model_encoder: nn.Module, encoder_checkpoint: str, device: str = "cpu"):
    """
    Загружает encoder_best.pt в encoder downstream-модели.

    Ожидаемый формат checkpoint:
    {
        'inc': state_dict(...),
        'down1': state_dict(...),
        'down2': state_dict(...),
        'down3': state_dict(...),
        'down4': state_dict(...),
    }
    """
    ckpt = torch.load(encoder_checkpoint, map_location=device)

    expected_keys = ["inc", "down1", "down2", "down3", "down4"]
    missing = [k for k in expected_keys if k not in ckpt]
    if len(missing) > 0:
        raise KeyError(f"Encoder checkpoint is missing keys: {missing}. Found keys: {list(ckpt.keys())}")

    model_encoder.inc.load_state_dict(ckpt["inc"], strict=True)
    model_encoder.down1.load_state_dict(ckpt["down1"], strict=True)
    model_encoder.down2.load_state_dict(ckpt["down2"], strict=True)
    model_encoder.down3.load_state_dict(ckpt["down3"], strict=True)
    model_encoder.down4.load_state_dict(ckpt["down4"], strict=True)

    return model_encoder

In [35]:
def build_model(
    scenario: str,
    device: str,
    encoder_checkpoint: str = None,
):
    """
    Собирает downstream-модель для сценариев:
    - scratch
    - ssl_ft

    scratch:
        encoder случайный

    ssl_ft:
        encoder той же архитектуры + загруженные SSL-веса

    head всегда создаётся заново
    """
    valid_scenarios = {"scratch", "ssl_ft"}
    if scenario not in valid_scenarios:
        raise ValueError(f"Unknown scenario: {scenario}. Expected one of {valid_scenarios}")

    # создаём базовый U-Net только как контейнер encoder-блоков
    unet = UNet1D_Light(
        n_channels=14,
        n_classes=14,
        base_ch=32,
        bilinear=True,
    )
    encoder = UNet1DEncoder(unet)

    if scenario == "ssl_ft":
        if encoder_checkpoint is None:
            raise ValueError("encoder_checkpoint must be provided for scenario='ssl_ft'")
        encoder = load_encoder_checkpoint_into_model_encoder(
            model_encoder=encoder,
            encoder_checkpoint=encoder_checkpoint,
            device=device,
        )

    head = ERPHead(in_features=512, num_classes=2)
    model = P300Model(encoder=encoder, head=head)
    model = model.to(device)

    return model

In [36]:
# Счётчик параметров
def count_all_parameters(model: nn.Module):
    return sum(p.numel() for p in model.parameters())


def count_trainable_parameters(model: nn.Module):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Поддержка 4 FT-сценариев

In [37]:
# ============================================================
# FT scenario helpers: freeze / unfreeze / optimizer groups
# ============================================================

import torch
import torch.nn as nn


def set_requires_grad(module: nn.Module, flag: bool) -> None:
    """Set requires_grad for all parameters in a module."""
    for p in module.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module) -> None:
    """Freeze all model parameters."""
    for p in model.parameters():
        p.requires_grad = False


def apply_trainable_mode(model: nn.Module, trainable_mode: str) -> None:
    """
    Configure which parts of the model are trainable.

    Expected model structure:
        model.encoder.inc
        model.encoder.down1
        model.encoder.down2
        model.encoder.down3
        model.encoder.down4
        model.head
    """
    freeze_all(model)

    if trainable_mode == "full":
        set_requires_grad(model.encoder, True)
        set_requires_grad(model.head, True)

    elif trainable_mode == "down4_only":
        set_requires_grad(model.encoder.down4, True)
        set_requires_grad(model.head, True)

    elif trainable_mode == "head_only":
        set_requires_grad(model.head, True)

    else:
        raise ValueError(f"Unknown trainable_mode: {trainable_mode}")


def get_trainable_parameter_names(model: nn.Module):
    """Return names of trainable parameters."""
    return [name for name, p in model.named_parameters() if p.requires_grad]


def count_parameters(model: nn.Module):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def build_ft_optimizer(
    model: nn.Module,
    scenario_name: str,
    lr_encoder: float,
    lr_head: float,
    weight_decay: float,
):
    """
    Build AdamW optimizer according to FT scenario config.

    Scenarios:
        - full_ft
        - low_lr_encoder
        - partial_ft
        - warmup
    """
    scenario_cfg = SCENARIO_CONFIGS[scenario_name]

    encoder_params = [p for p in model.encoder.parameters() if p.requires_grad]
    head_params = [p for p in model.head.parameters() if p.requires_grad]

    if not encoder_params and not head_params:
        raise ValueError("No trainable parameters found.")

    if scenario_cfg["use_discriminative_lr"]:
        param_groups = []
        if encoder_params:
            param_groups.append({
                "params": encoder_params,
                "lr": lr_encoder,
                "weight_decay": weight_decay,
            })
        if head_params:
            param_groups.append({
                "params": head_params,
                "lr": lr_head,
                "weight_decay": weight_decay,
            })
        optimizer = torch.optim.AdamW(param_groups)

    else:
        all_trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(
            all_trainable,
            lr=lr_head,
            weight_decay=weight_decay,
        )

    return optimizer


def summarize_trainable_parameters(model: nn.Module, max_lines: int = 50) -> None:
    """Print a short summary of trainable parameters."""
    total, trainable = count_parameters(model)
    names = get_trainable_parameter_names(model)

    print(f"Total params     : {total:,}")
    print(f"Trainable params : {trainable:,}")
    print(f"Frozen params    : {total - trainable:,}")
    print(f"Trainable tensors: {len(names)}")

    preview = names[:max_lines]
    if preview:
        print("\nTrainable parameter names:")
        for name in preview:
            print(" -", name)

    if len(names) > max_lines:
        print(f"... and {len(names) - max_lines} more")

## 8. Training / evaluation 

In [38]:
def train_one_epoch(model, loader, optimizer, criterion, device: str):
    """
    Одна эпоха обучения.
    Возвращает средний loss по эпохе.
    """
    if loader is None:
        raise ValueError("train loader is None")

    model.train()

    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        batch_size = xb.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

    epoch_loss = running_loss / max(n_samples, 1)
    return epoch_loss


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device: str):
    """
    Одна эпоха валидации.
    Возвращает средний loss по эпохе.
    """
    if loader is None:
        raise ValueError("val loader is None")

    model.eval()

    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        batch_size = xb.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

    epoch_loss = running_loss / max(n_samples, 1)
    return epoch_loss

In [39]:
# training history
def init_history():
    return {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
    }


def append_history(history: dict, epoch: int, train_loss: float, val_loss: float):
    history["epoch"].append(epoch)
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    return history

In [40]:
# цикл обучения
def fit_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device: str,
    max_epochs: int = 100,
    patience: int = 10,
    min_delta: float = 0.0,
    verbose: bool = True,
):
    """
    Train/val цикл с early stopping по val_loss.

    Возвращает словарь с:
    - history
    - best_epoch
    - best_val_loss
    - best_state_dict
    - stopped_epoch
    """
    if train_loader is None:
        raise ValueError("train_loader is None")
    if val_loader is None:
        raise ValueError("val_loader is None")

    history = init_history()
    early_stopper = EarlyStopping(
        patience=patience,
        min_delta=min_delta,
        mode="min",
    )

    stopped_epoch = max_epochs

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
        )

        val_loss = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        append_history(history, epoch, train_loss, val_loss)

        early_stopper.step(val_loss, model, epoch)

        if verbose:
            msg = (
                f"epoch {epoch:03d} | "
                f"train_loss={train_loss:.6f} | "
                f"val_loss={val_loss:.6f} | "
                f"best_val={early_stopper.best_value:.6f} @ epoch {early_stopper.best_epoch}"
            )
            print(msg)

        if early_stopper.should_stop:
            stopped_epoch = epoch
            if verbose:
                print(f"Early stopping triggered at epoch {epoch}.")
            break

    result = {
        "history": history,
        "best_epoch": early_stopper.best_epoch,
        "best_val_loss": early_stopper.best_value,
        "best_state_dict": early_stopper.best_state_dict,
        "stopped_epoch": stopped_epoch,
    }
    return result

In [41]:
# Optimizer и Loss

## Старый build_optimizer
# def build_optimizer(model, lr: float, weight_decay: float):
#     return torch.optim.AdamW(
#         filter(lambda p: p.requires_grad, model.parameters()),
#         lr=lr,
#         weight_decay=weight_decay,
#     )


def build_criterion(y_train=None):
    if y_train is None:
        return nn.CrossEntropyLoss()

    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()

    weight_pos = n_neg / (n_pos + 1e-8)

    class_weights = torch.tensor([1.0, weight_pos], dtype=torch.float32)

    return nn.CrossEntropyLoss(weight=class_weights.to(CONFIG["device"]))
    

In [42]:
# Загрузка best state
def load_best_model_state(model: nn.Module, fit_result: dict):
    """
    Загружает best_state_dict обратно в модель.
    """
    best_state_dict = fit_result.get("best_state_dict", None)
    if best_state_dict is None:
        raise ValueError("fit_result does not contain 'best_state_dict'")
    model.load_state_dict(best_state_dict)
    return model

## 10. Early stopping / checkpoint utilities

In [43]:
class EarlyStopping:
    """
    Early stopping по val_loss.

    mode='min' означает, что метрика должна уменьшаться.
    """
    def __init__(self, patience: int = 10, min_delta: float = 0.0, mode: str = "min"):
        if mode not in {"min", "max"}:
            raise ValueError("mode must be 'min' or 'max'")

        self.patience = patience
        self.min_delta = float(min_delta)
        self.mode = mode

        self.best_value = None
        self.best_epoch = None
        self.best_state_dict = None
        self.counter = 0
        self.should_stop = False

    def _is_improvement(self, value: float) -> bool:
        if self.best_value is None:
            return True

        if self.mode == "min":
            return value < (self.best_value - self.min_delta)
        else:
            return value > (self.best_value + self.min_delta)

    def step(self, value: float, model: nn.Module, epoch: int):
        """
        Обновляет состояние early stopping после очередной эпохи.
        """
        if self._is_improvement(value):
            self.best_value = float(value)
            self.best_epoch = int(epoch)
            self.best_state_dict = deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

## Test prediction и метрики

In [44]:
# prediction на loader
@torch.no_grad()
def predict_on_loader(model, loader, device: str):
    """
    Прогоняет модель по loader и возвращает:
    - y_true
    - prob_score   : probability of positive class
    - logit_score  : raw logit of positive class
    - y_pred       : threshold=0.5 on prob_score
    """
    if loader is None:
        raise ValueError("loader is None")

    model.eval()

    all_y = []
    all_prob = []
    all_logit = []
    all_pred = []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)

        logits = model(xb)                    # (B, 2)
        probs = torch.softmax(logits, dim=1)  # (B, 2)

        prob_score = probs[:, 1]
        logit_score = logits[:, 1]
        pred = (prob_score >= 0.5).long()

        all_y.append(yb.numpy())
        all_prob.append(prob_score.cpu().numpy())
        all_logit.append(logit_score.cpu().numpy())
        all_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(all_y) if len(all_y) > 0 else np.array([], dtype=np.int64)
    prob_score = np.concatenate(all_prob) if len(all_prob) > 0 else np.array([], dtype=np.float32)
    logit_score = np.concatenate(all_logit) if len(all_logit) > 0 else np.array([], dtype=np.float32)
    y_pred = np.concatenate(all_pred) if len(all_pred) > 0 else np.array([], dtype=np.int64)

    return {
        "y_true": y_true,
        "prob_score": prob_score,
        "logit_score": logit_score,
        "y_pred": y_pred,
    }

In [45]:
## FDR
def compute_fisher_fdr(y_true: np.ndarray, score: np.ndarray):
    """
    Fisher's Discriminant Ratio:
        FDR = (mu_pos - mu_neg)^2 / (var_pos + var_neg)

    score должен быть непрерывным classifier score.
    """
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    pos_scores = score[y_true == 1]
    neg_scores = score[y_true == 0]

    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return np.nan

    mu_pos = np.mean(pos_scores)
    mu_neg = np.mean(neg_scores)

    var_pos = np.var(pos_scores)
    var_neg = np.var(neg_scores)

    denom = var_pos + var_neg
    if denom <= 0:
        return np.nan

    return float((mu_pos - mu_neg) ** 2 / denom)

In [46]:
# accuracy, f1, precision, recall, fdr
def compute_metrics(
    y_true: np.ndarray,
    prob_score: np.ndarray,
    logit_score: np.ndarray,
    y_pred: np.ndarray,
):
    """
    Считает метрики в соответствии с логикой статьи:
    - AUC: по probability score
    - Accuracy: по binary predictions
    - F1: по binary predictions
    - Precision/Recall: по binary predictions
    - FDR: Fisher's Discriminant Ratio по classifier score
    """
    y_true = np.asarray(y_true).astype(int)
    prob_score = np.asarray(prob_score).astype(float)
    logit_score = np.asarray(logit_score).astype(float)
    y_pred = np.asarray(y_pred).astype(int)

    metrics = {}

    if len(np.unique(y_true)) < 2:
        metrics["auc"] = np.nan
    else:
        metrics["auc"] = float(roc_auc_score(y_true, prob_score))

    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    metrics["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    metrics["recall"] = float(recall_score(y_true, y_pred, zero_division=0))
    metrics["fdr"] = compute_fisher_fdr(y_true, logit_score)

    return metrics

In [47]:
# Объединяющая функия test evaluation
def evaluate_on_test(model, test_loader, device: str):
    pred_dict = predict_on_loader(
        model=model,
        loader=test_loader,
        device=device,
    )

    metrics = compute_metrics(
        y_true=pred_dict["y_true"],
        prob_score=pred_dict["prob_score"],
        logit_score=pred_dict["logit_score"],
        y_pred=pred_dict["y_pred"],
    )

    return {
        "predictions": pred_dict,
        "metrics": metrics,
    }

## Подбор порога

In [48]:
def select_best_threshold(y_true, prob_score, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.40, 71)

    best_t = 0.5
    best_f1 = -1.0

    for t in grid:
        y_pred = (prob_score >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)

    return best_t, best_f1

In [49]:
# Подсчёт метрик с заданным порогом
def compute_metrics_with_threshold(y_true, prob_score, logit_score, threshold):
    y_pred = (prob_score >= threshold).astype(int)
    return compute_metrics(
        y_true=y_true,
        prob_score=prob_score,
        logit_score=logit_score,
        y_pred=y_pred,
    )

## FT training wrappers

In [50]:
def train_standard_ft(
    model,
    train_loader,
    val_loader,
    criterion,
    device,
    scenario_name: str,
    lr_encoder: float,
    lr_head: float,
    weight_decay: float,
    max_epochs: int,
    patience: int,
    min_delta: float,
):
    """
    Standard FT training for:
        - full_ft
        - low_lr_encoder
        - partial_ft
    """
    scenario_cfg = SCENARIO_CONFIGS[scenario_name]

    apply_trainable_mode(model, scenario_cfg["trainable_mode"])

    optimizer = build_ft_optimizer(
        model=model,
        scenario_name=scenario_name,
        lr_encoder=lr_encoder,
        lr_head=lr_head,
        weight_decay=weight_decay,
    )

    stopper = EarlyStopping(
        patience=patience,
        min_delta=min_delta,
        mode="min",
    )

    history_rows = []
    stopped_epoch = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
        )

        val_loss = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        history_rows.append({
            "stage": "main",
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
        })

        stopper.step(val_loss, model, epoch)

        if stopper.should_stop:
            stopped_epoch = epoch
            break

    if stopped_epoch == 0:
        stopped_epoch = max_epochs

    return {
        "best_model_state": stopper.best_state_dict,
        "history_rows": history_rows,
        "best_epoch": stopper.best_epoch,
        "best_val_loss": stopper.best_value,
        "stopped_epoch": stopped_epoch,
    }


def train_warmup_ft(
    model,
    train_loader,
    val_loader,
    criterion,
    device,
    lr_encoder: float,
    lr_head: float,
    weight_decay: float,
    warmup_epochs: int,
    max_epochs: int,
    patience: int,
    min_delta: float,
):
    """
    Two-stage FT training:
        Stage 1: head-only warmup
        Stage 2: joint FT (full encoder + head with discriminative LR)
    """
    history_rows = []

    # -------------------------
    # Stage 1: head-only warmup
    # -------------------------
    apply_trainable_mode(model, "head_only")

    warmup_optimizer = build_ft_optimizer(
        model=model,
        scenario_name="warmup",
        lr_encoder=lr_encoder,
        lr_head=lr_head,
        weight_decay=weight_decay,
    )

    for epoch in range(1, warmup_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=warmup_optimizer,
            criterion=criterion,
            device=device,
        )

        val_loss = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        history_rows.append({
            "stage": "warmup",
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
        })

    # --------------------------------------
    # Stage 2: joint FT with early stopping
    # --------------------------------------
    apply_trainable_mode(model, "full")

    main_optimizer = build_ft_optimizer(
        model=model,
        scenario_name="warmup",
        lr_encoder=lr_encoder,
        lr_head=lr_head,
        weight_decay=weight_decay,
    )

    stopper = EarlyStopping(
        patience=patience,
        min_delta=min_delta,
        mode="min",
    )

    stopped_epoch = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=main_optimizer,
            criterion=criterion,
            device=device,
        )

        val_loss = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        history_rows.append({
            "stage": "main",
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
        })

        stopper.step(val_loss, model, epoch)

        if stopper.should_stop:
            stopped_epoch = epoch
            break

    if stopped_epoch == 0:
        stopped_epoch = max_epochs

    return {
        "best_model_state": stopper.best_state_dict,
        "history_rows": history_rows,
        "best_epoch": stopper.best_epoch,
        "best_val_loss": stopper.best_value,
        "stopped_epoch": stopped_epoch,
    }


def history_rows_to_df(history_rows):
    """Convert accumulated history rows to DataFrame."""
    if not history_rows:
        return pd.DataFrame(columns=["stage", "epoch", "train_loss", "val_loss"])
    return pd.DataFrame(history_rows)

## 11. Main run API

In [51]:
def extract_split_stats(indices_dict: dict, y: np.ndarray):
    def _count(idx):
        idx = np.asarray(idx, dtype=np.int64)
        n = len(idx)
        if n == 0:
            return {"n": 0, "n_pos": 0, "n_neg": 0}
        n_pos = int(y[idx].sum())
        n_neg = int(n - n_pos)
        return {"n": n, "n_pos": n_pos, "n_neg": n_neg}

    train_stats = _count(indices_dict["train_idx"])
    val_stats = _count(indices_dict["val_idx"])
    test_stats = _count(indices_dict["test_idx"])
    calib_stats = _count(indices_dict["calib_idx"])

    return {
        "n_calib": calib_stats["n"],
        "n_val": val_stats["n"],
        "n_test": test_stats["n"],
        "n_pos_calib": calib_stats["n_pos"],
        "n_pos_val": val_stats["n_pos"],
        "n_pos_test": test_stats["n_pos"],
    }

- p =0 → no training, fallback normalization with p=10

- early stopping by val_loss

- metrics include ROC-AUC, Accuracy, Binary F1, Fisher FDR

- same train/val split for scratch and ssl_ft

In [52]:
def run_one(
    subject_id: str,
    p: int,
    scenario: str,
    seed: int,
    group: str,
    encoder_checkpoint: str = None,
    ft_strategy: str = None,
    lr_encoder: float = 3e-5,
    lr_head: float = 3e-4,
    weight_decay: float = 1e-3,
    warmup_epochs: int = 3,
):
    """
    Один полный запуск pipeline для одного subject / p / scenario.

    Поддерживает:
    - scenario="scratch"
    - scenario="ssl_ft"

    Для scenario="ssl_ft" и p > 0 требуется ft_strategy:
    - "full_ft"
    - "low_lr_encoder"
    - "partial_ft"
    - "warmup"

    Для p > 0:
    - threshold подбирается на val по максимуму F1
    - затем применяется на test

    Возвращает словарь:
    - result_row   : итоговая строка результатов
    - history_df   : history по эпохам (или пустой df для p=0)
    - predictions  : y_true / prob_score / logit_score / y_pred
    """
    valid_scenarios = {"scratch", "ssl_ft"}
    valid_ft_strategies = {"full_ft", "low_lr_encoder", "partial_ft", "warmup"}

    if scenario not in valid_scenarios:
        raise ValueError(f"Unknown scenario: {scenario}. Expected one of {valid_scenarios}")

    if scenario == "ssl_ft" and encoder_checkpoint is None:
        raise ValueError("encoder_checkpoint must be provided for scenario='ssl_ft'")

    if scenario == "ssl_ft" and p > 0 and ft_strategy not in valid_ft_strategies:
        raise ValueError(
            f"For scenario='ssl_ft' and p > 0, ft_strategy must be one of {valid_ft_strategies}"
        )

    set_seed(seed)
    device = CONFIG["device"]

    # -------------------------------------------------
    # 1) load subject bundle
    # -------------------------------------------------
    bundle = load_subject_bundle(subject_id, p=p, group=group)

    # -------------------------------------------------
    # 2) prepare indices
    # -------------------------------------------------
    indices = prepare_run_indices(
        split=bundle["split"],
        y=bundle["y"],
        p=p,
        val_ratio=CONFIG["val_ratio"],
        seed=seed,
    )

    # -------------------------------------------------
    # 3) prepare normalized arrays
    # -------------------------------------------------
    arrays = prepare_indexed_arrays(
        bundle=bundle,
        indices_dict=indices,
        fallback_p_for_zero=CONFIG["fallback_p_for_zero"],
    )

    # -------------------------------------------------
    # 4) build loaders
    # -------------------------------------------------
    loaders = build_loaders(
        arrays_dict=arrays,
        batch_size=CONFIG["batch_size"],
        num_workers=CONFIG["num_workers"],
        pin_memory=CONFIG["pin_memory"],
    )

    # -------------------------------------------------
    # 5) build model
    # -------------------------------------------------
    model = build_model(
        scenario=scenario,
        device=device,
        encoder_checkpoint=encoder_checkpoint,
    )

    # Если у тебя build_criterion уже принимает y_train и считает class weights,
    # то оставь так. Если нет — верни criterion = build_criterion()
    criterion = build_criterion(arrays["y_train"]) if p > 0 else build_criterion()

    split_stats = extract_split_stats(indices, bundle["y"])

    # -------------------------------------------------
    # CASE A: p == 0 -> no training
    # -------------------------------------------------
    if p == 0:
        test_result = evaluate_on_test(
            model=model,
            test_loader=loaders["test_loader"],
            device=device,
        )
    
        result_row = {
            "subject_id": subject_id,
            "group": group,
            "p": p,
            "scenario": scenario,
            "ft_strategy": ft_strategy if scenario == "ssl_ft" else None,
            "seed": seed,
            "encoder_checkpoint": str(encoder_checkpoint) if encoder_checkpoint is not None else None,
            "lr_encoder": lr_encoder,
            "lr_head": lr_head,
            "weight_decay": weight_decay,
            "warmup_epochs": warmup_epochs if ft_strategy == "warmup" else None,
            "selected_threshold": 0.5,
            "val_f1_at_selected_threshold": None,
    
            **split_stats,
    
            "best_epoch": None,
            "best_val_loss": None,
            "stopped_epoch": None,
    
            "auc": test_result["metrics"]["auc"],
            "accuracy": test_result["metrics"]["accuracy"],
            "f1": test_result["metrics"]["f1"],
            "precision": test_result["metrics"]["precision"],
            "recall": test_result["metrics"]["recall"],
            "fdr": test_result["metrics"]["fdr"],
        }
    
        history_df = pd.DataFrame(columns=["stage", "epoch", "train_loss", "val_loss"])
    
        return {
            "result_row": result_row,
            "history_df": history_df,
            "predictions": test_result["predictions"],
        }

    # -------------------------------------------------
    # CASE B1: scratch -> old training logic
    # -------------------------------------------------
    if scenario == "scratch":
        optimizer = torch.optim.AdamW(
            [
                {"params": model.encoder.parameters(), "lr": lr_encoder},
                {"params": model.head.parameters(), "lr": lr_head},
            ],
            weight_decay=weight_decay,
        )
    
        fit_result = fit_model(
            model=model,
            train_loader=loaders["train_loader"],
            val_loader=loaders["val_loader"],
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            max_epochs=CONFIG["max_epochs"],
            patience=CONFIG["patience"],
            min_delta=CONFIG["min_delta"],
            verbose=False,
        )
    
        model = load_best_model_state(model, fit_result)
    
        val_pred = predict_on_loader(
            model=model,
            loader=loaders["val_loader"],
            device=device,
        )
    
        selected_threshold, val_f1_at_selected_threshold = select_best_threshold(
            y_true=val_pred["y_true"],
            prob_score=val_pred["prob_score"],
        )
    
        test_pred = predict_on_loader(
            model=model,
            loader=loaders["test_loader"],
            device=device,
        )
    
        test_metrics = compute_metrics_with_threshold(
            y_true=test_pred["y_true"],
            prob_score=test_pred["prob_score"],
            logit_score=test_pred["logit_score"],
            threshold=selected_threshold,
        )
    
        history_df = pd.DataFrame(fit_result["history"])
        if "stage" not in history_df.columns:
            history_df["stage"] = "main"
            history_df = history_df[["stage", "epoch", "train_loss", "val_loss"]]
    
        test_predictions = {
            "y_true": test_pred["y_true"],
            "prob_score": test_pred["prob_score"],
            "logit_score": test_pred["logit_score"],
            "y_pred": (test_pred["prob_score"] >= selected_threshold).astype(int),
        }
    
        result_row = {
            "subject_id": subject_id,
            "group": group,
            "p": p,
            "scenario": scenario,
            "ft_strategy": None,
            "seed": seed,
            "encoder_checkpoint": None,
            "lr_encoder": lr_encoder,
            "lr_head": lr_head,
            "weight_decay": weight_decay,
            "warmup_epochs": None,
            "selected_threshold": selected_threshold,
            "val_f1_at_selected_threshold": val_f1_at_selected_threshold,
    
            **split_stats,
    
            "best_epoch": fit_result["best_epoch"],
            "best_val_loss": fit_result["best_val_loss"],
            "stopped_epoch": fit_result["stopped_epoch"],
    
            "auc": test_metrics["auc"],
            "accuracy": test_metrics["accuracy"],
            "f1": test_metrics["f1"],
            "precision": test_metrics["precision"],
            "recall": test_metrics["recall"],
            "fdr": test_metrics["fdr"],
        }
    
        return {
            "result_row": result_row,
            "history_df": history_df,
            "predictions": test_predictions,
        }

    # -------------------------------------------------
    # CASE B2: ssl_ft -> Block 4 FT strategies
    # -------------------------------------------------
    if ft_strategy == "warmup":
        train_out = train_warmup_ft(
            model=model,
            train_loader=loaders["train_loader"],
            val_loader=loaders["val_loader"],
            criterion=criterion,
            device=device,
            lr_encoder=lr_encoder,
            lr_head=lr_head,
            weight_decay=weight_decay,
            warmup_epochs=warmup_epochs,
            max_epochs=CONFIG["max_epochs"],
            patience=CONFIG["patience"],
            min_delta=CONFIG["min_delta"],
        )
    else:
        train_out = train_standard_ft(
            model=model,
            train_loader=loaders["train_loader"],
            val_loader=loaders["val_loader"],
            criterion=criterion,
            device=device,
            scenario_name=ft_strategy,
            lr_encoder=lr_encoder,
            lr_head=lr_head,
            weight_decay=weight_decay,
            max_epochs=CONFIG["max_epochs"],
            patience=CONFIG["patience"],
            min_delta=CONFIG["min_delta"],
        )

    model.load_state_dict(train_out["best_model_state"])

    # ---------- val threshold selection ----------
    val_pred = predict_on_loader(
        model=model,
        loader=loaders["val_loader"],
        device=device,
    )

    selected_threshold, val_f1_at_selected_threshold = select_best_threshold(
        y_true=val_pred["y_true"],
        prob_score=val_pred["prob_score"],
    )

    # ---------- test evaluation with selected threshold ----------
    test_pred = predict_on_loader(
        model=model,
        loader=loaders["test_loader"],
        device=device,
    )

    test_metrics = compute_metrics_with_threshold(
        y_true=test_pred["y_true"],
        prob_score=test_pred["prob_score"],
        logit_score=test_pred["logit_score"],
        threshold=selected_threshold,
    )

    history_df = history_rows_to_df(train_out["history_rows"])

    test_predictions = {
        "y_true": test_pred["y_true"],
        "prob_score": test_pred["prob_score"],
        "logit_score": test_pred["logit_score"],
        "y_pred": (test_pred["prob_score"] >= selected_threshold).astype(int),
    }

    result_row = {
        "subject_id": subject_id,
        "group": group,
        "p": p,
        "scenario": scenario,
        "ft_strategy": ft_strategy,
        "seed": seed,
        "encoder_checkpoint": str(encoder_checkpoint) if encoder_checkpoint is not None else None,
        "lr_encoder": lr_encoder,
        "lr_head": lr_head,
        "weight_decay": weight_decay,
        "warmup_epochs": warmup_epochs if ft_strategy == "warmup" else None,
        "selected_threshold": selected_threshold,
        "val_f1_at_selected_threshold": val_f1_at_selected_threshold,

        **split_stats,

        "best_epoch": train_out["best_epoch"],
        "best_val_loss": train_out["best_val_loss"],
        "stopped_epoch": train_out["stopped_epoch"],

        "auc": test_metrics["auc"],
        "accuracy": test_metrics["accuracy"],
        "f1": test_metrics["f1"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "fdr": test_metrics["fdr"],
    }

    return {
        "result_row": result_row,
        "history_df": history_df,
        "predictions": test_predictions,
    }

In [53]:
# Для сохранения путей
def make_run_tag(subject_id: str, group: str, p: int, scenario: str, seed: int):
    return f"{group}__{subject_id}__p{p}__{scenario}__seed{seed}"


def ensure_results_dirs(results_root: Path):
    results_root = Path(results_root)
    (results_root / "history").mkdir(parents=True, exist_ok=True)
    (results_root / "predictions").mkdir(parents=True, exist_ok=True)
    (results_root / "tables").mkdir(parents=True, exist_ok=True)
    return results_root

In [54]:
# Сохранение hystory и prediction
def save_history_df(history_df: pd.DataFrame, run_tag: str, results_root: Path):
    results_root = ensure_results_dirs(results_root)
    out_path = results_root / "history" / f"{run_tag}.csv"
    history_df.to_csv(out_path, index=False)
    return out_path


def save_predictions_npz(predictions: dict, run_tag: str, results_root: Path):
    results_root = ensure_results_dirs(results_root)
    out_path = results_root / "predictions" / f"{run_tag}.npz"

    np.savez_compressed(
        out_path,
        y_true=predictions["y_true"],
        prob_score=predictions["prob_score"],
        logit_score=predictions["logit_score"],
        y_pred=predictions["y_pred"],
    )
    return out_path

In [55]:
# Запуск одно run с сохранением
def run_one_and_save(
    subject_id: str,
    p: int,
    scenario: str,
    seed: int,
    group: str,
    results_root: Path,
    encoder_checkpoint: str = None,
    lr_encoder: float = 3e-5,
    lr_head: float = 3e-4,
    weight_decay: float = 1e-3,
    warmup_epochs: int = 3,
    save_history: bool = True,
    save_predictions: bool = True,
):
    """
    Обёртка над run_one(...), которая:
    - запускает эксперимент
    - сохраняет history
    - сохраняет predictions
    - возвращает result_row
    """
    out = run_one(
        subject_id=subject_id,
        p=p,
        scenario=scenario,
        seed=seed,
        group=group,
        encoder_checkpoint=encoder_checkpoint,
        ft_strategy=None,
        lr_encoder=lr_encoder,
        lr_head=lr_head,
        weight_decay=weight_decay,
        warmup_epochs=warmup_epochs,
    )

    run_tag = make_run_tag(
        subject_id=subject_id,
        group=group,
        p=p,
        scenario=scenario,
        seed=seed,
    )

    history_path = None
    pred_path = None

    if save_history:
        history_path = save_history_df(
            history_df=out["history_df"],
            run_tag=run_tag,
            results_root=results_root,
        )

    if save_predictions:
        pred_path = save_predictions_npz(
            predictions=out["predictions"],
            run_tag=run_tag,
            results_root=results_root,
        )

    result_row = dict(out["result_row"])
    result_row["run_tag"] = run_tag
    result_row["history_path"] = str(history_path) if history_path is not None else None
    result_row["predictions_path"] = str(pred_path) if pred_path is not None else None

    return result_row

In [56]:
def run_many(
    subject_list,
    p_list,
    scenario_list,
    seed_list,
    group: str,
    results_root: Path,
    encoder_checkpoint: str = None,
    lr_encoder: float = 3e-5,
    lr_head: float = 3e-4,
    weight_decay: float = 1e-3,
    warmup_epochs: int = 3,
    save_history: bool = True,
    save_predictions: bool = True,
    save_summary_every: int = 1,
    continue_on_error: bool = True,
):
    """
    Массовый запуск серии экспериментов.

    Возвращает DataFrame с result_row по всем run.
    Промежуточно сохраняет summary_results.csv в /tables.
    """
    results_root = ensure_results_dirs(results_root)
    summary_path = results_root / "tables" / "summary_results.csv"

    rows = []
    run_counter = 0

    total_runs = (
        len(subject_list)
        * len(p_list)
        * len(scenario_list)
        * len(seed_list)
    )

    print(f"Planned runs: {total_runs}")

    for subject_id in subject_list:
        for p in p_list:
            for scenario in scenario_list:
                for seed in seed_list:
                    run_counter += 1
                    print(
                        f"[{run_counter}/{total_runs}] "
                        f"subject={subject_id} | group={group} | p={p} | scenario={scenario} | seed={seed}"
                    )

                    try:
                        row = run_one_and_save(
                            subject_id=subject_id,
                            p=p,
                            scenario=scenario,
                            seed=seed,
                            group=group,
                            results_root=results_root,
                            encoder_checkpoint=encoder_checkpoint if scenario == "ssl_ft" else None,
                            lr_encoder=lr_encoder,
                            lr_head=lr_head,
                            weight_decay=weight_decay,
                            warmup_epochs=warmup_epochs,
                            save_history=save_history,
                            save_predictions=save_predictions,
                        )
                        row["status"] = "ok"
                        row["error"] = None

                    except Exception as e:
                        if not continue_on_error:
                            raise

                        row = {
                            "subject_id": subject_id,
                            "group": group,
                            "p": p,
                            "scenario": scenario,
                            "seed": seed,
                            "encoder_checkpoint": str(encoder_checkpoint) if encoder_checkpoint is not None else None,
                            "status": "error",
                            "error": repr(e),
                        }
                        print(f"[ERROR] {row['error']}")

                    rows.append(row)

                    if run_counter % save_summary_every == 0:
                        pd.DataFrame(rows).to_csv(summary_path, index=False)

    results_df = pd.DataFrame(rows)
    results_df.to_csv(summary_path, index=False)
    return results_df

In [57]:
# Функция для чтения summary
def load_summary_results(results_root: Path):
    summary_path = Path(results_root) / "tables" / "summary_results.csv"
    if not summary_path.exists():
        raise FileNotFoundError(f"Summary file not found: {summary_path}")
    return pd.read_csv(summary_path)

## Новые обёртки для Dev-tuning

In [58]:
def make_run_tag_block4(
    subject_id: str,
    group: str,
    p: int,
    scenario: str,
    ft_strategy: str,
    seed: int,
    lr_encoder: float,
    lr_head: float,
    weight_decay: float,
):
    lr_enc_str = f"{lr_encoder:.0e}".replace("-", "m")
    lr_head_str = f"{lr_head:.0e}".replace("-", "m")
    wd_str = f"{weight_decay:.0e}".replace("-", "m")

    return (
        f"{subject_id}"
        f"__{group}"
        f"__p{p}"
        f"__{scenario}"
        f"__{ft_strategy}"
        f"__seed{seed}"
        f"__lre{lr_enc_str}"
        f"__lrh{lr_head_str}"
        f"__wd{wd_str}"
    )

In [59]:
def run_one_and_save_block4(
    subject_id: str,
    p: int,
    scenario: str,
    ft_strategy: str,
    seed: int,
    group: str,
    results_root: Path,
    encoder_checkpoint: str = None,
    lr_encoder: float = 3e-5,
    lr_head: float = 3e-4,
    weight_decay: float = 1e-3,
    warmup_epochs: int = 3,
    save_history: bool = True,
    save_predictions: bool = True,
):
    ensure_results_dirs(results_root)

    run_tag = make_run_tag_block4(
        subject_id=subject_id,
        group=group,
        p=p,
        scenario=scenario,
        ft_strategy=ft_strategy,
        seed=seed,
        lr_encoder=lr_encoder,
        lr_head=lr_head,
        weight_decay=weight_decay,
    )

    out = run_one(
        subject_id=subject_id,
        p=p,
        scenario=scenario,
        seed=seed,
        group=group,
        encoder_checkpoint=encoder_checkpoint,
        ft_strategy=ft_strategy,
        lr_encoder=lr_encoder,
        lr_head=lr_head,
        weight_decay=weight_decay,
        warmup_epochs=warmup_epochs,
    )

    result_row = out["result_row"]
    history_df = out["history_df"]
    predictions = out["predictions"]

    if save_history and history_df is not None and len(history_df) > 0:
        history_path = results_root / "history" / f"{run_tag}.csv"
        history_df.to_csv(history_path, index=False)

    if save_predictions and predictions is not None:
        save_predictions_npz(
            predictions=predictions,
            run_tag=run_tag,
            results_root=results_root,
        )

    return {
        "run_tag": run_tag,
        "result_row": result_row,
        "history_df": history_df,
        "predictions": predictions,
    }

In [60]:
def run_many_block4(
    subject_list,
    p_list,
    ft_strategy_list,
    seed_list,
    group: str,
    results_root: Path,
    encoder_checkpoint: str,
    scenario: str = "ssl_ft",
    lr_encoder: float = 3e-5,
    lr_head: float = 3e-4,
    weight_decay: float = 1e-3,
    warmup_epochs: int = 3,
    save_history: bool = True,
    save_predictions: bool = True,
    save_summary_every: int = 1,
    continue_on_error: bool = True,
):
    ensure_results_dirs(results_root)

    rows = []
    n_done = 0

    for subject_id in subject_list:
        for p in p_list:
            for ft_strategy in ft_strategy_list:
                for seed in seed_list:
                    print("=" * 90)
                    print(
                        f"RUN | subject={subject_id} | group={group} | p={p} | "
                        f"scenario={scenario} | ft_strategy={ft_strategy} | seed={seed}"
                    )

                    try:
                        out = run_one_and_save_block4(
                            subject_id=subject_id,
                            p=p,
                            scenario=scenario,
                            ft_strategy=ft_strategy,
                            seed=seed,
                            group=group,
                            results_root=results_root,
                            encoder_checkpoint=encoder_checkpoint,
                            lr_encoder=lr_encoder,
                            lr_head=lr_head,
                            weight_decay=weight_decay,
                            warmup_epochs=warmup_epochs,
                            save_history=save_history,
                            save_predictions=save_predictions,
                        )

                        rows.append(out["result_row"])
                        n_done += 1

                        if (n_done % save_summary_every) == 0:
                            summary_df = pd.DataFrame(rows)
                            summary_path = results_root / "tables" / "summary_results.csv"
                            summary_df.to_csv(summary_path, index=False)

                    except Exception as e:
                        print(f"[ERROR] subject={subject_id}, p={p}, ft_strategy={ft_strategy}, seed={seed}")
                        print(repr(e))

                        if not continue_on_error:
                            raise

    summary_df = pd.DataFrame(rows)
    summary_path = results_root / "tables" / "summary_results.csv"
    summary_df.to_csv(summary_path, index=False)

    return summary_df


## Проверки

In [61]:
import numpy as np
import torch

subject_id = "subjB"
group = "bcicomp3"
p = 10

bundle = load_subject_bundle(subject_id=subject_id, group=group, p=p)

print("bundle keys:", bundle.keys())
print("X:", bundle["X"].shape, bundle["X"].dtype)
print("y:", bundle["y"].shape, bundle["y"].dtype)
print("split keys:", bundle["split"].keys())
print("mean/std:", bundle["mean"].shape, bundle["std"].shape)

run_idx = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=0.2,
    seed=42,
)

print("\nrun_idx keys:", run_idx.keys())
for k, v in run_idx.items():
    if isinstance(v, (list, np.ndarray)):
        print(k, len(v))
    else:
        print(k, type(v))

bundle keys: dict_keys(['subject_id', 'group', 'p', 'X', 'y', 'split', 'mean', 'std'])
X: (15215, 14, 208) float32
y: (15215,) int8
split keys: dict_keys(['calib_pool_idx', 'test_rest_idx', 'calib_idx'])
mean/std: (14,) (14,)

run_idx keys: dict_keys(['calib_pool_idx', 'calib_idx', 'train_idx', 'val_idx', 'test_idx'])
calib_pool_idx 10651
calib_idx 1064
train_idx 851
val_idx 213
test_idx 4564


In [62]:
import numpy as np
import torch

subject_id = "subjB"
group = "bcicomp3"
p = 10
seed = 42

bundle = load_subject_bundle(subject_id=subject_id, group=group, p=p)

print("bundle keys:", bundle.keys())
print("X:", bundle["X"].shape, bundle["X"].dtype)
print("y:", bundle["y"].shape, bundle["y"].dtype)
print("split keys:", bundle["split"].keys())
print("mean/std:", bundle["mean"].shape, bundle["std"].shape)

indices_dict = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=0.2,
    seed=seed,
)

print("\nindices_dict keys:", indices_dict.keys())
for k, v in indices_dict.items():
    if isinstance(v, (list, np.ndarray)):
        print(f"{k}: n={len(v)}")
    else:
        print(f"{k}: type={type(v)}")

arrays_dict = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices_dict,
    fallback_p_for_zero=10,
)

print("\narrays_dict keys:", arrays_dict.keys())
for k, v in arrays_dict.items():
    if hasattr(v, "shape"):
        print(f"{k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"{k}: type={type(v)}")

loaders = build_loaders(
    arrays_dict=arrays_dict,
    batch_size=64,
    num_workers=0,
    pin_memory=True,
)

print("\nloader keys:", loaders.keys())
for name, loader in loaders.items():
    batch = next(iter(loader))
    xb, yb = batch
    print(f"{name}: x={xb.shape}, y={yb.shape}, x_dtype={xb.dtype}, y_dtype={yb.dtype}")

bundle keys: dict_keys(['subject_id', 'group', 'p', 'X', 'y', 'split', 'mean', 'std'])
X: (15215, 14, 208) float32
y: (15215,) int8
split keys: dict_keys(['calib_pool_idx', 'test_rest_idx', 'calib_idx'])
mean/std: (14,) (14,)

indices_dict keys: dict_keys(['calib_pool_idx', 'calib_idx', 'train_idx', 'val_idx', 'test_idx'])
calib_pool_idx: n=10651
calib_idx: n=1064
train_idx: n=851
val_idx: n=213
test_idx: n=4564

arrays_dict keys: dict_keys(['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test', 'mean', 'std'])
X_train: shape=(851, 14, 208), dtype=float32
y_train: shape=(851,), dtype=int8
X_val: shape=(213, 14, 208), dtype=float32
y_val: shape=(213,), dtype=int8
X_test: shape=(4564, 14, 208), dtype=float32
y_test: shape=(4564,), dtype=int8
mean: shape=(14,), dtype=float32
std: shape=(14,), dtype=float32

loader keys: dict_keys(['train_loader', 'val_loader', 'test_loader'])
train_loader: x=torch.Size([64, 14, 208]), y=torch.Size([64]), x_dtype=torch.float32, y_dtype=torch.int64
val

In [63]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(
    scenario="scratch",
    device=device,
    encoder_checkpoint=None,
)

xb, yb = next(iter(loaders["train_loader"]))
xb = xb.to(device)

with torch.no_grad():
    logits = model(xb)

print("logits.shape:", logits.shape)
print("expected:", (xb.shape[0], 2))
print("has_nan_logits:", torch.isnan(logits).any().item())

logits.shape: torch.Size([64, 2])
expected: (64, 2)
has_nan_logits: False


In [64]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(
    scenario="scratch",
    device=device,
    encoder_checkpoint=None,
)

trainable = [name for name, p in model.named_parameters() if p.requires_grad]
print("SCRATCH trainable params:", len(trainable))
print(*trainable[:50], sep="\n")

SCRATCH trainable params: 42
encoder.inc.block.0.weight
encoder.inc.block.0.bias
encoder.inc.block.1.weight
encoder.inc.block.1.bias
encoder.inc.block.3.weight
encoder.inc.block.3.bias
encoder.inc.block.4.weight
encoder.inc.block.4.bias
encoder.down1.block.1.block.0.weight
encoder.down1.block.1.block.0.bias
encoder.down1.block.1.block.1.weight
encoder.down1.block.1.block.1.bias
encoder.down1.block.1.block.3.weight
encoder.down1.block.1.block.3.bias
encoder.down1.block.1.block.4.weight
encoder.down1.block.1.block.4.bias
encoder.down2.block.1.block.0.weight
encoder.down2.block.1.block.0.bias
encoder.down2.block.1.block.1.weight
encoder.down2.block.1.block.1.bias
encoder.down2.block.1.block.3.weight
encoder.down2.block.1.block.3.bias
encoder.down2.block.1.block.4.weight
encoder.down2.block.1.block.4.bias
encoder.down3.block.1.block.0.weight
encoder.down3.block.1.block.0.bias
encoder.down3.block.1.block.1.weight
encoder.down3.block.1.block.1.bias
encoder.down3.block.1.block.3.weight
encode

In [65]:

model = build_model(
    scenario="ssl_ft",
    device=device,
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
)

# если partial_ft настраивается не в build_model, а позже в run_one/train-функциях,
# то тут может пока быть не финальная картина — но всё равно полезно посмотреть

trainable = [name for name, p in model.named_parameters() if p.requires_grad]
print("SSL_FT trainable params:", len(trainable))
print(*trainable[:80], sep="\n")

SSL_FT trainable params: 42
encoder.inc.block.0.weight
encoder.inc.block.0.bias
encoder.inc.block.1.weight
encoder.inc.block.1.bias
encoder.inc.block.3.weight
encoder.inc.block.3.bias
encoder.inc.block.4.weight
encoder.inc.block.4.bias
encoder.down1.block.1.block.0.weight
encoder.down1.block.1.block.0.bias
encoder.down1.block.1.block.1.weight
encoder.down1.block.1.block.1.bias
encoder.down1.block.1.block.3.weight
encoder.down1.block.1.block.3.bias
encoder.down1.block.1.block.4.weight
encoder.down1.block.1.block.4.bias
encoder.down2.block.1.block.0.weight
encoder.down2.block.1.block.0.bias
encoder.down2.block.1.block.1.weight
encoder.down2.block.1.block.1.bias
encoder.down2.block.1.block.3.weight
encoder.down2.block.1.block.3.bias
encoder.down2.block.1.block.4.weight
encoder.down2.block.1.block.4.bias
encoder.down3.block.1.block.0.weight
encoder.down3.block.1.block.0.bias
encoder.down3.block.1.block.1.weight
encoder.down3.block.1.block.1.bias
encoder.down3.block.1.block.3.weight
encoder

In [66]:
res_scratch_p10 = run_one(
    subject_id="subjB",
    p=10,
    scenario="scratch",
    seed=42,
    group="bcicomp3",
    encoder_checkpoint=None,
    ft_strategy=None,
    lr_encoder=3e-5,
    lr_head=3e-4,
    weight_decay=1e-3,
    warmup_epochs=3,
)

print(res_scratch_p10)

{'result_row': {'subject_id': 'subjB', 'group': 'bcicomp3', 'p': 10, 'scenario': 'scratch', 'ft_strategy': None, 'seed': 42, 'encoder_checkpoint': None, 'lr_encoder': 3e-05, 'lr_head': 0.0003, 'weight_decay': 0.001, 'warmup_epochs': None, 'selected_threshold': 0.39, 'val_f1_at_selected_threshold': 0.30434782608695654, 'n_calib': 1064, 'n_val': 213, 'n_test': 4564, 'n_pos_calib': 177, 'n_pos_val': 35, 'n_pos_test': 760, 'best_epoch': 5, 'best_val_loss': 0.680950489402377, 'stopped_epoch': 15, 'auc': 0.5347708782998505, 'accuracy': 0.37532865907099033, 'f1': 0.28095838587641864, 'precision': 0.17379095163806552, 'recall': 0.7328947368421053, 'fdr': 0.006141546040630407}, 'history_df':    stage  epoch  train_loss  val_loss
0   main      1    0.703212  0.694056
1   main      2    0.655956  0.693658
2   main      3    0.628197  0.683718
3   main      4    0.608348  0.687898
4   main      5    0.571825  0.680950
5   main      6    0.547447  0.681162
6   main      7    0.515350  0.734556
7   

In [67]:
res_partial_p10 = run_one(
    subject_id="subjB",
    p=10,
    scenario="ssl_ft",
    seed=42,
    group="bcicomp3",
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
    ft_strategy="partial_ft",
    lr_encoder=3e-5,
    lr_head=3e-4,
    weight_decay=1e-3,
    warmup_epochs=3,
)

print(res_partial_p10)

{'result_row': {'subject_id': 'subjB', 'group': 'bcicomp3', 'p': 10, 'scenario': 'ssl_ft', 'ft_strategy': 'partial_ft', 'seed': 42, 'encoder_checkpoint': '/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt', 'lr_encoder': 3e-05, 'lr_head': 0.0003, 'weight_decay': 0.001, 'warmup_epochs': None, 'selected_threshold': 0.39, 'val_f1_at_selected_threshold': 0.2834008097165992, 'n_calib': 1064, 'n_val': 213, 'n_test': 4564, 'n_pos_calib': 177, 'n_pos_val': 35, 'n_pos_test': 760, 'best_epoch': 11, 'best_val_loss': 0.6898667605270242, 'stopped_epoch': 21, 'auc': 0.5050642675300238, 'accuracy': 0.16739702015775634, 'f1': 0.2857142857142857, 'precision': 0.16666666666666666, 'recall': 1.0, 'fdr': 0.0004514092143962506}, 'history_df':    stage  epoch  train_loss  val_loss
0   main      1    0.701211  0.692686
1   main      2    0.697293  0.692563
2   main      3    0.692298  0.694375
3   main      4    0.694093  0.695605
4   main      5    0.690249  0.696147
5   main      

In [68]:
res_p0 = run_one(
    subject_id="subjB",
    p=0,
    scenario="scratch",
    seed=42,
    group="bcicomp3",
    encoder_checkpoint=None,
    ft_strategy=None,
    lr_encoder=3e-5,
    lr_head=3e-4,
    weight_decay=1e-3,
    warmup_epochs=3,
)

print(res_p0)

{'result_row': {'subject_id': 'subjB', 'group': 'bcicomp3', 'p': 0, 'scenario': 'scratch', 'ft_strategy': None, 'seed': 42, 'encoder_checkpoint': None, 'lr_encoder': 3e-05, 'lr_head': 0.0003, 'weight_decay': 0.001, 'warmup_epochs': None, 'selected_threshold': 0.5, 'val_f1_at_selected_threshold': None, 'n_calib': 0, 'n_val': 0, 'n_test': 4564, 'n_pos_calib': 0, 'n_pos_val': 0, 'n_pos_test': 760, 'best_epoch': None, 'best_val_loss': None, 'stopped_epoch': None, 'auc': 0.48266177569317614, 'accuracy': 0.8334794040315513, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'fdr': 0.001953746972027128}, 'history_df': Empty DataFrame
Columns: [stage, epoch, train_loss, val_loss]
Index: [], 'predictions': {'y_true': array([1, 0, 0, ..., 0, 0, 0]), 'prob_score': array([0.48614734, 0.48614502, 0.48614535, ..., 0.4861549 , 0.48615026,
       0.48614746], dtype=float32), 'logit_score': array([-0.02663162, -0.02663563, -0.02663191, ..., -0.02661795,
       -0.02663402, -0.02663174], dtype=float32), 'y_pre

## Полный прогон FT-стратегий на Test

In [69]:
# # =========================
# # Smoke test: FT on benchmark
# # =========================

# FT_SMOKE_DIR = FINAL_TEST_FT_DIR / "smoke_test"
# FT_SMOKE_DIR.mkdir(parents=True, exist_ok=True)

# FT_SMOKE_SUBJECTS = [FINAL_TEST_CONFIG["subjects"][0]]
# FT_SMOKE_P_LIST = [0, 10, 20]
# FT_SMOKE_STRATEGIES = ["full_ft", "partial_ft"]
# FT_SMOKE_SEEDS = [42]

# print("FT_SMOKE_DIR:", FT_SMOKE_DIR)
# print("subject:", FT_SMOKE_SUBJECTS[0])

# ft_smoke_df = run_many_block4(
#     subject_list=FT_SMOKE_SUBJECTS,
#     p_list=FT_SMOKE_P_LIST,
#     ft_strategy_list=FT_SMOKE_STRATEGIES,
#     seed_list=FT_SMOKE_SEEDS,
#     group=FINAL_TEST_CONFIG["group"],
#     results_root=FT_SMOKE_DIR,
#     encoder_checkpoint=CONFIG["encoder_checkpoint"],
#     scenario="ssl_ft",
#     lr_encoder=FINAL_TEST_CONFIG["lr_encoder"],
#     lr_head=FINAL_TEST_CONFIG["lr_head"],
#     weight_decay=FINAL_TEST_CONFIG["weight_decay"],
#     warmup_epochs=FINAL_TEST_CONFIG["warmup_epochs"],
#     save_history=True,
#     save_predictions=True,
#     save_summary_every=1,
#     continue_on_error=False,
# )

# display(ft_smoke_df)

# print("Unique p:", sorted(ft_smoke_df["p"].unique().tolist()))
# print("Unique strategies:", sorted(ft_smoke_df["ft_strategy"].dropna().unique().tolist()))

# expected_ft_smoke_rows = (
#     len(FT_SMOKE_SUBJECTS)
#     * len(FT_SMOKE_P_LIST)
#     * len(FT_SMOKE_STRATEGIES)
#     * len(FT_SMOKE_SEEDS)
# )

# print("Expected rows:", expected_ft_smoke_rows)
# print("Actual rows:", len(ft_smoke_df))
# print("Matches expected:", len(ft_smoke_df) == expected_ft_smoke_rows)

In [70]:
final_test_ft_df = run_many_block4(
    subject_list=['subjB'],
    p_list=FINAL_TEST_CONFIG["p_list"],
    ft_strategy_list=FINAL_TEST_CONFIG["ft_strategy_list"],
    seed_list=FINAL_TEST_CONFIG["seed_list"],
    group="bcicomp3",
    results_root=FINAL_TEST_FT_DIR,
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
    scenario="ssl_ft",
    lr_encoder=FINAL_TEST_CONFIG["lr_encoder"],
    lr_head=FINAL_TEST_CONFIG["lr_head"],
    weight_decay=FINAL_TEST_CONFIG["weight_decay"],
    warmup_epochs=FINAL_TEST_CONFIG["warmup_epochs"],
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

final_test_ft_summary_path = FINAL_TEST_FT_DIR / "summary.csv"
final_test_ft_df.to_csv(final_test_ft_summary_path, index=False)

print("FT test run finished.")
print("Saved to:", final_test_ft_summary_path)
print("Shape:", final_test_ft_df.shape)
display(final_test_ft_df.head())

## Полный прогон Scratch на Test

RUN | subject=subjB | group=bcicomp3 | p=0 | scenario=ssl_ft | ft_strategy=full_ft | seed=42
RUN | subject=subjB | group=bcicomp3 | p=0 | scenario=ssl_ft | ft_strategy=low_lr_encoder | seed=42
RUN | subject=subjB | group=bcicomp3 | p=0 | scenario=ssl_ft | ft_strategy=partial_ft | seed=42
RUN | subject=subjB | group=bcicomp3 | p=0 | scenario=ssl_ft | ft_strategy=warmup | seed=42
RUN | subject=subjB | group=bcicomp3 | p=10 | scenario=ssl_ft | ft_strategy=full_ft | seed=42
RUN | subject=subjB | group=bcicomp3 | p=10 | scenario=ssl_ft | ft_strategy=low_lr_encoder | seed=42
RUN | subject=subjB | group=bcicomp3 | p=10 | scenario=ssl_ft | ft_strategy=partial_ft | seed=42
RUN | subject=subjB | group=bcicomp3 | p=10 | scenario=ssl_ft | ft_strategy=warmup | seed=42
RUN | subject=subjB | group=bcicomp3 | p=20 | scenario=ssl_ft | ft_strategy=full_ft | seed=42
RUN | subject=subjB | group=bcicomp3 | p=20 | scenario=ssl_ft | ft_strategy=low_lr_encoder | seed=42
RUN | subject=subjB | group=bcicomp3 | 

,subject_id,group,p,scenario,ft_strategy,seed,encoder_checkpoint,lr_encoder,lr_head,weight_decay,...,n_pos_test,best_epoch,best_val_loss,stopped_epoch,auc,accuracy,f1,precision,recall,fdr
0,subjB,bcicomp3,0,ssl_ft,full_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,760,NaN,NaN,NaN,0.509543,0.166521,0.2855,0.166521,1.0,0.001064
1,subjB,bcicomp3,0,ssl_ft,low_lr_encoder,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,760,NaN,NaN,NaN,0.509543,0.166521,0.2855,0.166521,1.0,0.001064
2,subjB,bcicomp3,0,ssl_ft,partial_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,760,NaN,NaN,NaN,0.509543,0.166521,0.2855,0.166521,1.0,0.001064
3,subjB,bcicomp3,0,ssl_ft,warmup,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,760,NaN,NaN,NaN,0.509543,0.166521,0.2855,0.166521,1.0,0.001064
4,subjB,bcicomp3,10,ssl_ft,full_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0.00003,0.0003,0.001,...,760,2.0,0.695601,12.0,0.497778,0.166521,0.2855,0.166521,1.0,0.000024


In [71]:
# # =========================
# # Smoke test: scratch on benchmark
# # =========================

# SCRATCH_SMOKE_DIR = FINAL_TEST_SCRATCH_DIR / "smoke_test"
# SCRATCH_SMOKE_DIR.mkdir(parents=True, exist_ok=True)

# SCRATCH_SMOKE_SUBJECTS = [FINAL_TEST_SCRATCH_CONFIG["subjects"][0]]
# SCRATCH_SMOKE_P_LIST = [0, 10, 20]
# SCRATCH_SMOKE_SCENARIOS = ["scratch"]
# SCRATCH_SMOKE_SEEDS = [42]

# print("SCRATCH_SMOKE_DIR:", SCRATCH_SMOKE_DIR)
# print("subject:", SCRATCH_SMOKE_SUBJECTS[0])

# scratch_smoke_df = run_many(
#     subject_list=SCRATCH_SMOKE_SUBJECTS,
#     p_list=SCRATCH_SMOKE_P_LIST,
#     scenario_list=SCRATCH_SMOKE_SCENARIOS,
#     seed_list=SCRATCH_SMOKE_SEEDS,
#     group=FINAL_TEST_SCRATCH_CONFIG["group"],
#     results_root=SCRATCH_SMOKE_DIR,
#     encoder_checkpoint=None,
#     save_history=True,
#     save_predictions=True,
#     save_summary_every=1,
#     continue_on_error=False,
# )

# display(scratch_smoke_df)

# print("Unique p:", sorted(scratch_smoke_df["p"].unique().tolist()))
# print("Unique scenarios:", sorted(scratch_smoke_df["scenario"].dropna().unique().tolist()))

# expected_scratch_smoke_rows = (
#     len(SCRATCH_SMOKE_SUBJECTS)
#     * len(SCRATCH_SMOKE_P_LIST)
#     * len(SCRATCH_SMOKE_SCENARIOS)
#     * len(SCRATCH_SMOKE_SEEDS)
# )

# print("Expected rows:", expected_scratch_smoke_rows)
# print("Actual rows:", len(scratch_smoke_df))
# print("Matches expected:", len(scratch_smoke_df) == expected_scratch_smoke_rows)

In [72]:
# =========================
# Full run: scratch (single-subject, BCI Competition III Dataset II)
# =========================

final_test_scratch_df = run_many(
    subject_list=["subjB"],   # ["subjB"]
    p_list=FINAL_TEST_SCRATCH_CONFIG["p_list"],           # [0,10,20,40,60,100]
    scenario_list=["scratch"],
    seed_list=[42],     # [42]
    group="bcicomp3",             # "bcicomp3"
    results_root=FINAL_TEST_SCRATCH_DIR,
    encoder_checkpoint=None,                              # важно для scratch
    lr_encoder=FINAL_TEST_SCRATCH_CONFIG["lr_encoder"],
    lr_head=FINAL_TEST_SCRATCH_CONFIG["lr_head"],
    weight_decay=FINAL_TEST_SCRATCH_CONFIG["weight_decay"],
    warmup_epochs=3,                                      # не используется, но можно оставить
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

final_test_scratch_summary_path = FINAL_TEST_SCRATCH_DIR / "summary.csv"
final_test_scratch_df.to_csv(final_test_scratch_summary_path, index=False)

print("\nScratch single-subject run finished.")
print("Saved to:", final_test_scratch_summary_path)
print("Shape:", final_test_scratch_df.shape)

display(final_test_scratch_df.head())

Planned runs: 6
[1/6] subject=subjB | group=bcicomp3 | p=0 | scenario=scratch | seed=42
[2/6] subject=subjB | group=bcicomp3 | p=10 | scenario=scratch | seed=42
[3/6] subject=subjB | group=bcicomp3 | p=20 | scenario=scratch | seed=42
[4/6] subject=subjB | group=bcicomp3 | p=40 | scenario=scratch | seed=42
[5/6] subject=subjB | group=bcicomp3 | p=60 | scenario=scratch | seed=42
[6/6] subject=subjB | group=bcicomp3 | p=100 | scenario=scratch | seed=42

Scratch single-subject run finished.
Saved to: /kaggle/working/stage5_block2_results/stage5_final_eval_test_SubjB_all_strategies/scratch/summary.csv
Shape: (6, 33)


,subject_id,group,p,scenario,ft_strategy,seed,encoder_checkpoint,lr_encoder,lr_head,weight_decay,...,accuracy,f1,precision,recall,fdr,run_tag,history_path,predictions_path,status,error
0,subjB,bcicomp3,0,scratch,None,42,None,0.00003,0.0003,0.001,...,0.833479,0.000000,0.000000,0.000000,0.001954,bcicomp3__subjB__p0__scratch__seed42,/kaggle/working/stage5_block2_results/stage5_f...,/kaggle/working/stage5_block2_results/stage5_f...,ok,None
1,subjB,bcicomp3,10,scratch,None,42,None,0.00003,0.0003,0.001,...,0.375329,0.280958,0.173791,0.732895,0.006142,bcicomp3__subjB__p10__scratch__seed42,/kaggle/working/stage5_block2_results/stage5_f...,/kaggle/working/stage5_block2_results/stage5_f...,ok,None
2,subjB,bcicomp3,20,scratch,None,42,None,0.00003,0.0003,0.001,...,0.166521,0.285500,0.166521,1.000000,0.003930,bcicomp3__subjB__p20__scratch__seed42,/kaggle/working/stage5_block2_results/stage5_f...,/kaggle/working/stage5_block2_results/stage5_f...,ok,None
3,subjB,bcicomp3,40,scratch,None,42,None,0.00003,0.0003,0.001,...,0.248247,0.289207,0.171625,0.918421,0.008086,bcicomp3__subjB__p40__scratch__seed42,/kaggle/working/stage5_block2_results/stage5_f...,/kaggle/working/stage5_block2_results/stage5_f...,ok,None
4,subjB,bcicomp3,60,scratch,None,42,None,0.00003,0.0003,0.001,...,0.299080,0.287369,0.172969,0.848684,0.008091,bcicomp3__subjB__p60__scratch__seed42,/kaggle/working/stage5_block2_results/stage5_f...,/kaggle/working/stage5_block2_results/stage5_f...,ok,None
